## INM713 Semantic Web Technologies and Knowledge Graphs 

### Coursework Project (Part 2): Ontology Alignment (Task OA)

In [1]:
# Matthew Lau (250060300)

### Deafult Setting 

In [2]:
# Install libraries
!pip install owlready2==0.45
!pip install rdflib==7.1.3 
!pip install python-Levenshtein==0.26.1 
!pip install owlrl==7.1.3 
!pip install pandas==2.2.3 
!pip install sentence-transformers

In [3]:
# Import Libraries
from rdflib import Graph, URIRef, Namespace
from rdflib.namespace import OWL, RDF, RDFS

# Source: Laboratory 5 support code by Ernesto Jiménez-Ruiz
# Repository:https://github.com/city-knowledge-graphs/python-2026/tree/f3a1ce17323e95a45d253b1a5fcc1ad4a3fa806d/lab5
import Levenshtein as lev
from isub import isub

import time
import os
import pandas as pd
import requests
import json

In [4]:
# Base path for all OAEI datasets
OAEI_BASE = os.path.normpath(os.path.expanduser("~/Desktop/OAEI-INM713-IN3067"))

# Conference dataset paths
CONFERENCE_ONTO_PATH = os.path.normpath(os.path.join(OAEI_BASE, "conference", "ontologies"))
CONFERENCE_REF_PATH = os.path.normpath(os.path.join(OAEI_BASE, "conference"))

# MLOA Base path
MLOA_BASE = os.path.normpath(os.path.expanduser("~/Desktop/MLOA"))

# Output directories for each subtask
OA1_DIR = os.path.normpath(os.path.join(MLOA_BASE, "Ontology_Alignments", "OA.1"))
OA2_DIR = os.path.normpath(os.path.join(MLOA_BASE, "Ontology_Alignments", "OA.2"))
OA3_DIR = os.path.normpath(os.path.join(MLOA_BASE, "Ontology_Alignments", "OA.3"))
OA4_DIR = os.path.normpath(os.path.join(MLOA_BASE, "Ontology_Alignments", "OA.4"))

# Create directories
os.makedirs(OA1_DIR, exist_ok=True)
os.makedirs(OA2_DIR, exist_ok=True)
os.makedirs(OA3_DIR, exist_ok=True)
os.makedirs(OA4_DIR, exist_ok=True)

# output_dir points to OA.4 (backward compatible)
output_dir = OA4_DIR

print(f" OA.1: {OA1_DIR}")
print(f" OA.2: {OA2_DIR}")
print(f" OA.3: {OA3_DIR}")
print(f" OA.4: {OA4_DIR}")

 OA.1: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1
 OA.2: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.2
 OA.3: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.3
 OA.4: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4


In [5]:
# Loading Ontologies and Extracting Labels
def getLabel(entity):
    " Extract the best available label for an ontology entity."
    if len(entity.label) > 0:
        return str(entity.label[0]) # Use rdfs:label if available
    else:
        return str(entity.name) # Otherwise use URI name
        
def loadOntology(path):
    g = Graph()
    
    if path.endswith(".ttl"):
        g.parse(path, format="turtle")
    else:
        g.parse(path, format="xml") 
    return g
    
def getClasses(g):
    """
    Extract all OWL classes from an ontology graph.
    """
    classes = []
    for s in g.subjects(RDF.type, OWL.Class):
        if isinstance(s, URIRef):  # exclude blank nodes
            classes.append(s)
    return classes
    
def getObjectProperties(g):
    """
    Extract all OWL object properties from an ontology graph.
    """
    props = []
    for s in g.subjects(RDF.type, OWL.ObjectProperty):
        if isinstance(s, URIRef):
            props.append(s)
    return props

def getDataProperties(g):
    """
    Extract all OWL data properties from an ontology graph.
    """
    props = []
    for s in g.subjects(RDF.type, OWL.DatatypeProperty):
        if isinstance(s, URIRef):
            props.append(s)
    return props
    
def getIndividuals(g):
    """
    Extract all OWL named individuals from an ontology graph.
    Supports both owl:NamedIndividual and direct rdf:type assertions.
    """
    system_namespaces = {
        "http://www.w3.org/2002/07/owl#",
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "http://www.w3.org/2000/01/rdf-schema#",
        "http://www.w3.org/2001/XMLSchema#",
        "http://www.w3.org/2004/02/skos/core#",
    }
    # Filter image and video file
    image_extensions = {
        '.jpg', '.jpeg', '.png', '.gif',
        '.svg', '.webp', '.pdf', '.mp3',
        '.mp4', '.wav', '.ogg'
    }
    
    individuals = set()

    # Method 1：owl:NamedIndividual
    for s in g.subjects(RDF.type, OWL.NamedIndividual):
        if isinstance(s, URIRef):
            individuals.add(s)

    # Method 2：rdf:type 
    classes = set(getClasses(g))
    for s, p, o in g.triples(
        (None, RDF.type, None)
    ):
        if not isinstance(s, URIRef):
            continue
        if not isinstance(o, URIRef):
            continue
        # filter namespaces  
        o_str = str(o)
        if any(o_str.startswith(ns)
               for ns in system_namespaces):
            continue
            
        # filter picture
        s_str = str(s).lower()
        if any(ext in s_str
               for ext in image_extensions):
            continue

        # filter classes
        if s in classes:
            continue
            
        individuals.add(s)

    return list(individuals)


def getLabel(g, entity):
    """
    Extract the best available label for an ontology entity.
    """
    for label in g.objects(entity, RDFS.label):
        return str(label)
        
    uri = str(entity)
    if "#" in uri:
        return uri.split("#")[-1]   
    else:
        return uri.split("/")[-1] 
        
def getOntologyInfo(g, onto_name="Ontology"):
    """
    Print a summary of an ontology.
    """
    classes = getClasses(g)
    obj_props = getObjectProperties(g)
    data_props = getDataProperties(g)
    individuals = getIndividuals(g)

    print(f"=== {onto_name} Summary ===")
    print(f"Classes:            {len(classes)}")
    print(f"Object properties:  {len(obj_props)}")
    print(f"Data properties:    {len(data_props)}")
    print(f"Individuals:        {len(individuals)}")

    print(f"\nFirst 5 classes and their labels:")
    for cls in classes[:5]:
        print(f"  URI:   {cls}")
        print(f"  Label: {getLabel(g, cls)}")
        print()

In [6]:
# Test the function 
cmt_path = os.path.join(CONFERENCE_ONTO_PATH, "cmt.owl")
ekaw_path = os.path.join(CONFERENCE_ONTO_PATH, "ekaw.owl")

print("Loading ontologies")
g1 = loadOntology(cmt_path)
g2 = loadOntology(ekaw_path)
print(f"{g1,g2}\nBoth ontologies loaded successfully!\n")

getOntologyInfo(g1, "cmt.owl")
print()
getOntologyInfo(g2, "ekaw.owl")

Loading ontologies
(<Graph identifier=N7e3b2941a1fb4ea989117dae8a16f269 (<class 'rdflib.graph.Graph'>)>, <Graph identifier=N6494fa44c6354f88997ee378994255f3 (<class 'rdflib.graph.Graph'>)>)
Both ontologies loaded successfully!

=== cmt.owl Summary ===
Classes:            29
Object properties:  49
Data properties:    10
Individuals:        0

First 5 classes and their labels:
  URI:   http://cmt#Meta-Reviewer
  Label: Meta-Reviewer

  URI:   http://cmt#Reviewer
  Label: Reviewer

  URI:   http://cmt#Decision
  Label: Decision

  URI:   http://cmt#Person
  Label: Person

  URI:   http://cmt#Document
  Label: Document


=== ekaw.owl Summary ===
Classes:            73
Object properties:  33
Data properties:    0
Individuals:        0

First 5 classes and their labels:
  URI:   http://ekaw#Event
  Label: Event

  URI:   http://ekaw#Person
  Label: Person

  URI:   http://ekaw#Accepted_Paper
  Label: Accepted_Paper

  URI:   http://ekaw#Rejected_Paper
  Label: Rejected_Paper

  URI:   http:/

### Subtask OA.1 (Give the system a name)

In [7]:
# System name: Matthew Lau Ontology Aligner (MLOA)
# To produce alignments with two different ontologies in Turtle format by combing lexical, structural and LLM-based techniques

### Subtask OA.1 (use one base alignment technique)

In [8]:
# Step 1: basic alignment technique - Lexical Matching - Class Matching
# Lexical matching compares the labels of entities from two ontologies using string similarity, such as exacting mathing and Isub similarity.

In [9]:
# Source: "A String Metric for Ontology Alignment" from Australian National University
# Repository:https://sourceforge.net/projects/febrl/

def normaliseLabel(label):
    """
    Normalise a label for comparison:
    1. Convert to lowercase
    2. Remove underscores and hyphens
    3. Strip whitespace

    Example:
        "hasConferenceChair" → "hasconferencechair"
        "conference_chair"   → "conferencechair"
    """
    label = label.lower()
    label = label.replace("_", "").replace("-", "")
    label = label.strip()
    return label

def lexicalSimilarity(label1, label2):
    """
    Compute similarity between two labels.
    Returns a score between 0.0 and 1.0.

    Uses ISub algorithm designed specifically for ontology alignment.
    """
    # Normalise both labels first
    norm1 = normaliseLabel(label1)
    norm2 = normaliseLabel(label2)

    # Exact match → score 1.0
    if norm1 == norm2:
        return 1.0

    # ISub similarity for non-exact matches
    return isub(norm1, norm2)

def lexicalMatcher(g1, g2, threshold=0.9):
    """
    Basic lexical matcher: compare all classes from two ontologies.
    Returns a list of mappings (entity1, entity2, score).

    Parameters:
        g1: source ontology
        g2 : target ontology
        threshold : minimum similarity score to accept a mapping (default 0.9 — only very similar labels)
    """
    mappings = []

    # Get all classes from both ontologies
    classes1 = getClasses(g1)
    classes2 = getClasses(g2)

    print(f"Comparing {len(classes1)} classes "
          f"vs {len(classes2)} classes")
    print(f"Total comparisons: "
          f"{len(classes1) * len(classes2)}\n")

     # Compare every class in g1 against every class in g2
    for c1 in classes1:
        label1 = getLabel(g1, c1)

        for c2 in classes2:
            label2 = getLabel(g2, c2)

            # Calculate similarity
            score = lexicalSimilarity(label1, label2)

            # Only keep mappings above threshold
            if score >= threshold:
                mappings.append((c1, c2, score))

    print(f"Found {len(mappings)} mappings "
          f"above threshold {threshold}")
    return mappings

In [10]:
# Run Lexical Matching on cmt.owl vs ekaw.owl

In [11]:
start_time = time.time()

mappings = lexicalMatcher(g1, g2, threshold=0.9)

end_time = time.time()
elapsed = round(end_time - start_time, 2)

print(f"\nComputation time: {elapsed} seconds")
print(f"\nSample mappings found:")
print(f"{'Source':<40} {'Target':<40} {'Score'}")
print("-" * 85)

for src, tgt, score in mappings[:10]:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<40} {tgt_label:<40} {score:.3f}")

Comparing 29 classes vs 73 classes
Total comparisons: 2117

Found 12 mappings above threshold 0.9

Computation time: 0.12 seconds

Sample mappings found:
Source                                   Target                                   Score
-------------------------------------------------------------------------------------
Reviewer                                 Review                                   0.957
Person                                   Person                                   1.000
Document                                 Document                                 1.000
Conference                               Conference_Paper                         0.940
Conference                               Conference_Session                       0.922
Conference                               Conference_Trip                          0.950
Conference                               Conference                               1.000
Conference                               Conference_Banq

In [12]:
# One-to-One mapping constraint to modify the accuracy for basic Lexical Matching
# It means each entity can only be matched to its best candidate
def lexicalMatcherOneToOne(g1, g2, threshold=0.9):
    """
    Improved lexical matcher with one-to-one mapping constraint.
    Each entity in g1 is matched to at most ONE entity in g2
    (the one with the highest similarity score).
    """
    mappings = []
    classes1 = getClasses(g1)
    classes2 = getClasses(g2)

    print(f"Comparing {len(classes1)} classes "
          f"vs {len(classes2)} classes")

    for c1 in classes1:
        label1  = getLabel(g1, c1)
        best_score = 0.0
        best_match = None

        for c2 in classes2:
            label2 = getLabel(g2, c2)
            score  = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = c2

        if best_match is not None:
            mappings.append((c1, best_match, best_score))

    print(f"Found {len(mappings)} mappings "
          f"above threshold {threshold}")
    return mappings

In [13]:
# Compare basic lexical matching vs one-to-one mapping
print("=== Basic lexical matching  ===")
mappings_original = lexicalMatcher(g1, g2, threshold=0.9)

print("\n=== One-to-One Matcher ===")
mappings_improved = lexicalMatcherOneToOne(g1, g2, threshold=0.9)

print("\n=== Comparison ===")
print(f"Original mappings:  {len(mappings_original)}")
print(f"Improved mappings:  {len(mappings_improved)}")

print(f"\nImproved mappings detail:")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in mappings_improved:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

=== Basic lexical matching  ===
Comparing 29 classes vs 73 classes
Total comparisons: 2117

Found 12 mappings above threshold 0.9

=== One-to-One Matcher ===
Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9

=== Comparison ===
Original mappings:  12
Improved mappings:  8

Improved mappings detail:
Source                              Target                              Score
---------------------------------------------------------------------------
Reviewer                            Review                              0.957
Person                              Person                              1.000
Document                            Document                            1.000
Conference                          Conference                          1.000
ConferenceChair                     Conference                          0.940
ConferenceMember                    Conference                          0.931
Review                              Review              

In [14]:
# Step 2: basic alignment technique - Property Matching

In [15]:
# Extends lexical matching to object and data properties
# Source: Laboratory 5 support code "AccessEntityLabels.pyby Ernesto Jiménez-Ruiz
# Repository:https://github.com/city-knowledge-graphs/python-2026/blob/f3a1ce17323e95a45d253b1a5fcc1ad4a3fa806d/lab5/AccessEntityLabels.py

def propertyMatcher(g1, g2, threshold=0.9):
    """
    Match object properties and data properties between two ontologies using lexical similarity.
    Uses one-to-one mapping constraint: each property matched to its best candidate only.
    """
    # Object Properties 
    obj_props1 = getObjectProperties(g1)
    obj_props2 = getObjectProperties(g2)

    print(f"Object properties: "
          f"{len(obj_props1)} vs {len(obj_props2)}")
    print(f"Total comparisons: "
          f"{len(obj_props1) * len(obj_props2)}")

    obj_mappings = []
    for p1 in obj_props1:
        label1 = getLabel(g1, p1)
        best_score = 0.0
        best_match = None

        for p2 in obj_props2:
            label2 = getLabel(g2, p2)
            score = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = p2

        if best_match is not None:
            obj_mappings.append((p1, best_match, best_score))

    print(f"Object property mappings found: "
          f"{len(obj_mappings)}\n")

    # Data Properties
    data_props1 = getDataProperties(g1)
    data_props2 = getDataProperties(g2)

    print(f"Data properties: "
          f"{len(data_props1)} vs {len(data_props2)}")
    print(f"Total comparisons: "
          f"{len(data_props1) * len(data_props2)}")

    data_mappings = []
    for p1 in data_props1:
        label1     = getLabel(g1, p1)
        best_score = 0.0
        best_match = None

        for p2 in data_props2:
            label2 = getLabel(g2, p2)
            score  = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = p2

        if best_match is not None:
            data_mappings.append((p1, best_match, best_score))

    print(f"Data property mappings found: "
          f"{len(data_mappings)}")

    return obj_mappings, data_mappings


In [16]:
# Run property matcher waon cmt.owl vs ekaw.owl
start_time = time.time()

obj_mappings, data_mappings = propertyMatcher(g1, g2, threshold=0.9)

end_time = time.time()
elapsed  = round(end_time - start_time, 2)

print(f"\nComputation time: {elapsed} seconds")

print(f"\n=== Object Property Mappings ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in obj_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

print(f"\n=== Data Property Mappings ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in data_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 1

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0

Computation time: 0.13 seconds

=== Object Property Mappings ===
Source                              Target                              Score
---------------------------------------------------------------------------
writtenBy                           writtenBy                           1.000

=== Data Property Mappings ===
Source                              Target                              Score
---------------------------------------------------------------------------


In [17]:
# Try different thresholds for property matching

for threshold in [0.9, 0.8, 0.7, 0.6]:
    obj_m, data_m = propertyMatcher(g1, g2, threshold=threshold)
    print(f"Threshold {threshold}: "
          f"{len(obj_m)} object property mappings\n")

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 1

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.9: 1 object property mappings

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.8: 7 object property mappings

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 22

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.7: 22 object property mappings

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 38

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Threshold 0.6: 38 object property mappings



In [18]:
# Test threshold 0.8 property mappings
obj_mappings_08, _ = propertyMatcher(g1, g2, threshold=0.8)

print(f"\n=== Object Property Mappings (threshold=0.8) ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)

for src, tgt, score in obj_mappings_08:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0

=== Object Property Mappings (threshold=0.8) ===
Source                              Target                              Score
---------------------------------------------------------------------------
assignedByReviewer                  hasReviewer                         0.821
assignReviewer                      hasReviewer                         0.883
reviewCriteriaEnteredBy             reviewWrittenBy                     0.871
endReview                           reviewWrittenBy                     0.803
hasAuthor                           authorOf                            0.801
writtenBy                           writtenBy                           1.000
writeReview                         reviewWrittenBy                     0.865


In [19]:
# Base on the above result, set the threshold of classes = 0.9, threshold of properties = 0.8

In [20]:
# Step 3: Saving alignments in Turtle Format
 # owl:equivalentClass (class mappings)
 # owl:equivalentProperty (property mappings)

SKOS_CONCEPT = URIRef("http://www.w3.org/2004/02/skos/core#Concept")

def isSKOSConcept(g, entity):
    """Check if entity is a SKOS Concept."""
    return (entity, RDF.type, SKOS_CONCEPT) in g

def saveMappings(class_mappings, obj_mappings, data_mappings, output_path, individual_mappings=[], equiv_threshold=0.9, g1=None, g2=None):
    """
    Save all mappings as RDF triples in Turtle format.
    
    Parameters:
        class_mappings  : list of (class1, class2, score)
        obj_mappings    : list of (prop1, prop2, score)
        data_mappings   : list of (prop1, prop2, score)
        output_path     : path to save the .ttl file
        
    Predicate selected based on similarity score:
      score >= 0.9 → owl:equivalentClass
      score <  0.9 → rdfs:subClassOf
    """
    # Create a new RDF graph for the alignment
    alignment = Graph()

    # Bind namespaces for cleaner output
    alignment.bind("owl", OWL)
    alignment.bind("rdf", RDF)
    alignment.bind("rdfs", RDFS)

    # Class Mappings
    for c1, c2, score in class_mappings:
        # SKOS Concepts → owl:sameAs
        if g1 and isSKOSConcept(g1, c1):
            pred = OWL.sameAs
        # High similarity → owl:equivalentClass
        elif score >= equiv_threshold:
            pred = OWL.equivalentClass
        # Lower similarity → rdfs:subClassOf
        else:
            pred = RDFS.subClassOf

        alignment.add((URIRef(c1), pred, URIRef(c2)))
        
    # Object Property Mappings
    for p1, p2, score in obj_mappings:
        if score >= equiv_threshold:
            pred = OWL.equivalentProperty
        else:
            pred = RDFS.subPropertyOf
        alignment.add((URIRef(p1), pred, URIRef(p2)))

    # Data propoerty Mappings
    for p1, p2, score in data_mappings:
        if score >= equiv_threshold:
            pred = OWL.equivalentProperty
        else:
            pred = RDFS.subPropertyOf
        alignment.add((URIRef(p1), pred,URIRef(p2)))
        
    # Instance mappings
    for i1, i2, score in individual_mappings:
        alignment.add((URIRef(i1), OWL.sameAs, URIRef(i2)))

    # Saving 
    alignment.serialize(destination=output_path, format="turtle")

    # Summary
    equiv_c = sum(1 for _,_,s in class_mappings
                  if s >= equiv_threshold)
    sub_c   = sum(1 for _,_,s in class_mappings
                  if s < equiv_threshold)
    equiv_p = sum(1 for _,_,s in
                  (obj_mappings + data_mappings)
                  if s >= equiv_threshold)
    sub_p   = sum(1 for _,_,s in
                  (obj_mappings + data_mappings)
                  if s < equiv_threshold)


    print(f" Alignment saved to: {output_path}")
    print(f"  owl:equivalentClass:    {equiv_c}")
    print(f"  rdfs:subClassOf:        {sub_c}")
    print(f"  owl:equivalentProperty: {equiv_p}")
    print(f"  rdfs:subPropertyOf:     {sub_p}")
    print(f"  owl:sameAs:             {len(individual_mappings)}")
    print(f"  Total:                  "
          f"{equiv_c+sub_c+equiv_p+sub_p+len(individual_mappings)}")


In [21]:
# File name follows coursework convention:
# mysystem-onto1-onto2.ttl
output_file = os.path.join(OA1_DIR, "mloa-cmt-ekaw.ttl")

# Get class mappings (one-to-one, threshold 0.9)
class_mappings = lexicalMatcherOneToOne(g1, g2, threshold=0.9)

# Get property mappings (threshold 0.8)
obj_mappings, data_mappings = propertyMatcher(g1, g2, threshold=0.8)

# Save all mappings
saveMappings(class_mappings, obj_mappings, data_mappings, output_file,g1=g1, g2=g2)

Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9
Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw.ttl
  owl:equivalentClass:    8
  rdfs:subClassOf:        0
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     6
  owl:sameAs:             0
  Total:                  15


In [22]:
# Step 4:  Precision (Pr), Recall(Re), and F-score (F) to evaluate an alignment 
# - Precision = correct mappings / total computed mappings
# - Recall    = correct mappings / total reference mappings  
# - F-score   = harmonic mean of Precision and Recall

In [23]:
# Source: Lab 5 support code "oa_notebook.ipynb" by Ernesto Jiménez-Ruiz
# Repository: https://github.com/city-knowledge-graphs/python-2026/blob/f3a1ce17323e95a45d253b1a5fcc1ad4a3fa806d/lab5/oa_notebook.ipynb
def evaluate(reference_file, system_file):
    """
    Compare system alignment against reference alignment.
    Computes Precision, Recall and F-score.
    """
    # Load reference alignment
    ref = Graph()
    with open(reference_file, 'r', encoding='utf-8') as f:
        ref.parse(f, format="turtle")

    # Load system alignment
    sys_graph = Graph()
    with open(system_file, 'r', encoding='utf-8') as f:
        sys_graph.parse(f, format="turtle")

    # Calculate TP, FP, FN
    tp = 0  # Correct mappings
    fp = 0  # Wrong mappings in system but not in reference
    fn = 0  # Missed mappings in reference but not in system

    # Check each system mapping against reference
    for s, p, o in sys_graph:
        if (s, p, o) in ref or (o, p, s) in ref:
            tp += 1  # Correct
        else:
            fp += 1  # Incorrect

    # Check each reference mapping against system
    for s, p, o in ref:
        if (s, p, o) not in sys_graph and \
           (o, p, s) not in sys_graph:
            fn += 1  # Missed
            
    # Calculate metrics
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f_score = (2 * precision * recall) / \
                (precision + recall) \
                if (precision + recall) > 0 else 0
    
    print(f"Evaluating: {os.path.basename(system_file)}")
    print(f"Reference:  {os.path.basename(reference_file)}")
    print(f"─" * 45)
    print(f"True Positives  (TP): {tp}")
    print(f"False Positives (FP): {fp}")
    print(f"False Negatives (FN): {fn}")
    print(f"─" * 45)
    print(f"Precision: {precision:.3f}")
    print(f"Recall:    {recall:.3f}")
    print(f"F-score:   {f_score:.3f}")

    return precision, recall, f_score

In [24]:
# Evalute my system alignment vs reference alignment
# Reference alignment path
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment path
system_file = output_file  # mloa-cmt-ekaw.ttl

# Run evaluation
precision, recall, f_score = evaluate(reference_file, system_file)

Evaluating: mloa-cmt-ekaw.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 10
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.333
Recall:    0.455
F-score:   0.385


In [25]:
# Results Tracking Initialisation
results = {"Task": [],"Technique": [],"Precision": [],"Recall": [],"F-score": [],"Time(s)": []}

In [26]:
# Step 5: Result Tracking of my MLOA system
# Record improvement after each techinque used

# Record first result
results["Task"].append("cmt-ekaw")
results["Technique"].append("Lexical (threshold=0.9/0.8)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed, 3))

df_results = pd.DataFrame(results)

# Display as table
print("=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35} "
          f"{row['Precision']:>10.3f} {row['Recall']:>8.3f} "
          f"{row['F-score']:>8.3f} {row['Time(s)']:>8.2f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385     0.13


### Subtask OA.1 (Combine one or more techniques)

In [27]:
# Step 1: Combined alignment technique - Structural Matching 
# Extend lexical matching with structural information. Use parent class relationships to improve class mapping. 
# For example, if two classess have smiliar names and both have smiliar parent classes, then the mapping is considered more reliable.

In [28]:
def getParentClasses(g, entity):
    """
    Get all parent classes of an entity via rdfs:subClassOf.
    """
    parents = []
    for parent in g.objects(entity, RDFS.subClassOf):
        if isinstance(parent, URIRef):
            parents.append(parent)
    return parents

def structuralScore(g1, g2, c1, c2, lexical_mappings):
    """
    Calculate structural similarity between two classes.
    Logic:
    - Get parent classes of c1 and c2
    - Check if any parent pairs are already matched in lexical_mappings

    Parameters:
        g1: rdflib Graph (source ontology)
        g2: rdflib Graph (target ontology)
        c1: parent classes from g1)
        c2: parent classes from g2)
        lexical_mappings: list of (class1, class2, score) from lexical matcher
    """
    parents1 = getParentClasses(g1, c1)
    parents2 = getParentClasses(g2, c2)

    if not parents1 or not parents2:
        return 0.0

    # Set up lookup set of lexical mappings 
    matched_pairs = set()
    for m1, m2, _ in lexical_mappings:
        matched_pairs.add((str(m1), str(m2)))
        matched_pairs.add((str(m2), str(m1)))

    # Check whether patent class is matched
    for p1 in parents1:
        for p2 in parents2:
            if (str(p1), str(p2)) in matched_pairs:
                return 1.0  

    return 0.0  

def structuralMatcher(g1, g2,lexical_threshold=0.9,combined_threshold=0.7):
    """
    Combined lexical + structural for classes.

    Scoring formula:
        combined_score = (lexical_score * 0.6) + (structural_score * 0.4)

    Weights explanation:
        Lexical:    60% weight (primary signal)
        Structural: 40% weight (supporting signal)
    """
    classes1 = getClasses(g1)
    classes2 = getClasses(g2)

    # print(f"Running structural matching.")
    # print(f"Classes: {len(classes1)} vs {len(classes2)}")

    # Computing progress
    lexical_mappings = []
    for c1 in classes1:
        label1 = getLabel(g1, c1)
        best_score = 0.0
        best_match = None

        for c2 in classes2:
            label2 = getLabel(g2, c2)
            score  = lexicalSimilarity(label1, label2)

            if score >= lexical_threshold and score > best_score:
                best_score = score
                best_match = c2

        if best_match is not None:
            lexical_mappings.append((c1, best_match, best_score))

    #print(f"Lexical mappings found: {len(lexical_mappings)}")

    # Combined score（Lexical + Structural）
    final_mappings = []

    for c1 in classes1:
        label1 = getLabel(g1, c1)
        best_score = 0.0
        best_match = None

        for c2 in classes2:
            label2 = getLabel(g2, c2)

            # Lexical score
            lex_score  = lexicalSimilarity(label1, label2)

            # Structural score
            struct_score = structuralScore(g1, g2, c1, c2, lexical_mappings)

            # Combined score
            combined = (lex_score * 0.6) + (struct_score * 0.4)

            if combined >= combined_threshold and combined > best_score:
                best_score = combined
                best_match = c2

        if best_match is not None:
            final_mappings.append((c1, best_match,
                                   best_score))

    #print(f"Final mappings found: {len(final_mappings)}")
    return final_mappings

In [29]:
# Run structural matching on cmt.owl vs ekaw.owl

start_time = time.time()

struct_mappings = structuralMatcher(g1, g2, lexical_threshold = 0.9, combined_threshold = 0.7)

end_time = time.time()
elapsed_struct = round(end_time - start_time, 2)

print(f"\nComputation time: {elapsed_struct} seconds")

# Result
print(f"\n=== Structural Matcher Results ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)

for src, tgt, score in struct_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")


Computation time: 0.29 seconds

=== Structural Matcher Results ===
Source                              Target                              Score
---------------------------------------------------------------------------
Meta-Reviewer                       Neutral_Review                      0.755
ConferenceMember                    Conference_Participant              0.851
ExternalReviewer                    Possible_Reviewer                   0.767
Review                              Review                              1.000
Paper                               Paper                               1.000
PaperAbstract                       Poster_Paper                        0.830
PaperFullVersion                    Conference_Paper                    0.814
Meta-Review                         Neutral_Review                      0.785


In [30]:
# Step 2: Merge lexical + structural mappings
def mergeMapping(lexical_mappings, structural_mappings):
    """
    Merge lexical and structural mappings.
    
    Strategy:
    - Start with lexical mappings as base and add structural mappings that found NEW entities not already covered by lexical matcher
    - If same entity mapped differently, keep higher score
    
    """
    merged = {}

    # lexical mappings
    for c1, c2, score in lexical_mappings:
        merged[str(c1)] = (c1, c2, score)

    # structural mappings
    for c1, c2, score in structural_mappings:
        key = str(c1)

        if key not in merged:
            merged[key] = (c1, c2, score)
        else:
            existing_score = merged[key][2]
            if score > existing_score:
                merged[key] = (c1, c2, score)

    result = list(merged.values())
    # print(f"Lexical mappings:    {len(lexical_mappings)}")
    # print(f"Structural mappings: {len(structural_mappings)}")
    # print(f"Merged mappings:     {len(result)}")
    return result

In [31]:
# Step 3: Run Merge lexical + structural mappings
lexical_class_mappings = lexicalMatcherOneToOne(g1, g2, threshold=0.9)

# Merge
print("=== Merging Lexical + Structural ===")
merged_class_mappings = mergeMapping(lexical_class_mappings,struct_mappings)

print(f"\n=== Merged Mappings Detail ===")
print(f"{'Source':<35} {'Target':<35} {'Score'}")
print("-" * 75)
for src, tgt, score in merged_class_mappings:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<35} {tgt_label:<35} {score:.3f}")

# Save merged alignment
merged_file = os.path.join(OA1_DIR, "mloa-cmt-ekaw-merged.ttl")
saveMappings(merged_class_mappings, obj_mappings, data_mappings, merged_file, g1=g1, g2=g2)
print(f" Saved: mloa-cmt-ekaw-merged.ttl")

Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9
=== Merging Lexical + Structural ===

=== Merged Mappings Detail ===
Source                              Target                              Score
---------------------------------------------------------------------------
Reviewer                            Review                              0.957
Person                              Person                              1.000
Document                            Document                            1.000
Conference                          Conference                          1.000
ConferenceChair                     Conference                          0.940
ConferenceMember                    Conference                          0.931
Review                              Review                              1.000
Paper                               Paper                               1.000
Meta-Reviewer                       Neutral_Review                      0.755
Ext

In [32]:
# Step 4: Evaluate: Lexical + Structural (merged)
# Reference alignment
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment
merged_file = os.path.join(OA1_DIR, "mloa-cmt-ekaw-merged.ttl")

# Run evaluation
precision, recall, f_score = evaluate(reference_file, merged_file)

Evaluating: mloa-cmt-ekaw-merged.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 15
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.250
Recall:    0.455
F-score:   0.323


In [33]:
# Step 5: Result tracking (Lexical + Structural)

results["Task"].append("cmt-ekaw")
results["Technique"].append("Lexical + Structural (merged)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed_struct, 3))

# Display table
df_results = pd.DataFrame(results)
print(f"=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.130
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.290


### Subtask OA.1 (improve scalability)

In [34]:
# Step 1: Scalability technique- Inverted Index
# Tokenise all labels to words and only compare classes sharing common tokens

In [35]:
# Soruce: LogMap: Logic-based and Scalable Ontology Matching. ISWC 2011 by Ernesto Jiménez-Ruiz
# Repository: https://github.com/ernestojimenezruiz/logmap-matcher

import re
from collections import defaultdict

def tokenise(label):
    """
    Split a label into tokens (individual words).
    Handles camelCase, underscores, and hyphens.
    """

    label = re.sub(r'([A-Z])', r' \1', label)

    label = label.replace('_', ' ').replace('-', ' ')

    tokens = label.lower().split()

    tokens = [t for t in tokens if len(t) > 2]

    return tokens
    
def buildIndex(g):
    """
    Build an inverted index for all classes and properties in an ontology.
    Index structure:
        token → [list of entities containing this token]
    Example:
        "paper"  → [ekaw:Paper, ekaw:Conference_Paper]
        "review" → [ekaw:Review, ekaw:Neutral_Review]
    """
    index = defaultdict(list)

    # Index classes
    for entity in getClasses(g):
        label  = getLabel(g, entity)
        tokens = tokenise(label)
        for token in tokens:
            index[token].append(entity)

    # Index object properties
    for entity in getObjectProperties(g):
        label  = getLabel(g, entity)
        tokens = tokenise(label)
        for token in tokens:
            index[token].append(entity)

    # Index data properties
    for entity in getDataProperties(g):
        label  = getLabel(g, entity)
        tokens = tokenise(label)
        for token in tokens:
            index[token].append(entity)

    return index

def getCandidates(g1, g2, index2):
    """
    Use inverted index to find candidate matches, only compare entities that share at least one common token.
    """
    candidates = defaultdict(set)
    total_candidates = 0

    all_entities1 = (list(getClasses(g1)) + list(getObjectProperties(g1)) + list(getDataProperties(g1)))

    for entity1 in all_entities1:
        label = getLabel(g1, entity1)
        tokens = tokenise(label)

        for token in tokens:
            if token in index2:
                for entity2 in index2[token]:
                    candidates[entity1].add(entity2)

        total_candidates += len(candidates[entity1])

    return candidates, total_candidates

def indexedMatcher(g1, g2, threshold=0.9):
    """
    Scalable matcher using inverted index.
    """
    # Create index
    print("Building inverted index")
    index2 = buildIndex(g2)
    print(f"Index built: {len(index2)} unique tokens")

    # Searching
    print("Finding candidates")
    candidates, total_candidates = getCandidates(g1, g2, index2)

    # Compute how many saved
    all_entities1 = (list(getClasses(g1)) + list(getObjectProperties(g1)) + list(getDataProperties(g1)))

    all_entities2 = (list(getClasses(g2)) + list(getObjectProperties(g2)) +list(getDataProperties(g2)))

    brute_force = len(all_entities1) * len(all_entities2)

    print(f"\nScalability comparison:")
    print(f"  Brute force comparisons: {brute_force:,}")
    print(f"  Index-based comparisons: {total_candidates:,}")
    if brute_force > 0:
        reduction = (1 - total_candidates/brute_force) * 100
        print(f"  Reduction: {reduction:.1f}%")

    # Compare chosen pairs only
    print(f"\nMatching candidates")
    mappings = []

    for entity1, entity2_candidates in candidates.items():
        label1 = getLabel(g1, entity1)
        best_score = 0.0
        best_match = None

        for entity2 in entity2_candidates:
            label2 = getLabel(g2, entity2)
            score = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = entity2

        if best_match is not None:
            mappings.append((entity1, best_match, best_score))

    print(f"Mappings found: {len(mappings)}")
    return mappings


In [36]:
# Test: Tokenise functinon
test_labels = ["ConferenceChair", "Conference_Paper", "hasAuthor","writtenBy", "Meta-Reviewer", "Paper"]

print("=== Tokenise Test ===")
for label in test_labels:
    tokens = tokenise(label)
    print(f"  {label:<25} → {tokens}")

=== Tokenise Test ===
  ConferenceChair           → ['conference', 'chair']
  Conference_Paper          → ['conference', 'paper']
  hasAuthor                 → ['has', 'author']
  writtenBy                 → ['written']
  Meta-Reviewer             → ['meta', 'reviewer']
  Paper                     → ['paper']


In [37]:
# Step 2: Test Inverted Index Matcher on cmt.owl vs ekaw.owl
print("=" * 50)
print("BRUTE FORCE MATCHER")
print("=" * 50)
start = time.time()
bf_class = lexicalMatcherOneToOne(g1, g2, threshold=0.9)
bf_obj, bf_data = propertyMatcher(g1, g2, threshold=0.8)
elapsed_bf = round(time.time() - start, 3)
total_bf = (len(list(getClasses(g1))) * len(list(getClasses(g2)))) + (len(list(getObjectProperties(g1))) * len(list(getObjectProperties(g2))))
print(f"Total comparisons: {total_bf:,}")
print(f"Mappings found:    {len(bf_class) + len(bf_obj)}")
print(f"Time:              {elapsed_bf}s")

print()
print("=" * 50)
print("INDEXED MATCHER")
print("=" * 50)
start_time = time.time()
indexed_mappings = indexedMatcher(g1, g2, threshold=0.9)
elapsed_idx = round(time.time() - start_time, 3)
print(f"Time: {elapsed_idx}s")

idx_class_mappings = [(s, t, sc) for s, t, sc in indexed_mappings if (s, RDF.type, OWL.Class) in g1]
idx_obj_mappings = [(s, t, sc) for s, t, sc in indexed_mappings if (s, RDF.type, OWL.ObjectProperty) in g1]
idx_data_mappings = [(s, t, sc) for s, t, sc in indexed_mappings if (s, RDF.type, OWL.DatatypeProperty) in g1]

print(f"\nClass mappings:    {len(idx_class_mappings)}")
print(f"Object prop:       {len(idx_obj_mappings)}")
print(f"Data prop:         {len(idx_data_mappings)}")

# Save indexed alignment
indexed_file = os.path.join(OA1_DIR, "mloa-cmt-ekaw-indexed.ttl")
saveMappings(idx_class_mappings, idx_obj_mappings, idx_data_mappings, indexed_file, g1=g1, g2=g2)

print(f"Saved: mloa-cmt-ekaw-indexed.ttl")
print()
print("=" * 50)
print("COMPARISON")
print("=" * 50)
if elapsed_bf > 0:
    speedup = round(elapsed_bf / elapsed_idx, 1) \
              if elapsed_idx > 0 else "N/A"
    print(f"Brute force time:  {elapsed_bf}s")
    print(f"Indexed time:      {elapsed_idx}s")
    print(f"Speed improvement: {speedup}x faster")

BRUTE FORCE MATCHER
Comparing 29 classes vs 73 classes
Found 8 mappings above threshold 0.9
Object properties: 49 vs 33
Total comparisons: 1617
Object property mappings found: 7

Data properties: 10 vs 0
Total comparisons: 0
Data property mappings found: 0
Total comparisons: 3,734
Mappings found:    15
Time:              0.249s

INDEXED MATCHER
Building inverted index
Index built: 85 unique tokens
Finding candidates

Scalability comparison:
  Brute force comparisons: 9,328
  Index-based comparisons: 539
  Reduction: 94.2%

Matching candidates
Mappings found: 16
Time: 0.037s

Class mappings:    9
Object prop:       5
Data prop:         2
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-indexed.ttl
  owl:equivalentClass:    9
  rdfs:subClassOf:        0
  owl:equivalentProperty: 7
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  16
Saved: mloa-cmt-ekaw-indexed.ttl

COMPARISON
Brute force time:  0.249s
Indexed time:  

In [38]:
# Step 3:The evaluation metrics of the Indexed Matcher
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment
indexed_file = os.path.join(OA1_DIR, "mloa-cmt-ekaw-indexed.ttl")

# Run evaluation
precision, recall, f_score = evaluate(reference_file, indexed_file)

Evaluating: mloa-cmt-ekaw-indexed.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 11
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.312
Recall:    0.455
F-score:   0.370


In [39]:
# Step 4: Results Tracking (Indexed Matcher result)
results["Task"].append("cmt-ekaw")
results["Technique"].append("Indexed Matcher")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed_idx, 3))

# Display table
df_results = pd.DataFrame(results)
print(f"=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.130
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.290
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.037


### Subtask OA.1 (advanced alignment techniques - Embedding & LLM)

#### Advanced technique 1: Embedding Matching

In [40]:
# Step1: Embedding is to capture semantic similarity between ontology labels rather then just doing pure string matching between entities

In [41]:
# Model: all-MiniLM-L6-v2 (sentence-transformers)

In [42]:
# Source: Sentence Embeddings using Siamese BERT-Networks (EMNLP 2019) by Nils Reimers, Iryna Gurevych
# Repository: https://arxiv.org/abs/1908.10084
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Install SentenceTransformer(
print("Loading embedding model")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(" Model loaded")

# Create function
def getEmbedding(labels):
    """
    Convert a list of labels into embedding vectors.
    """
    return model.encode(labels)

def embeddingSimilarity(embedding1, embedding2):
    """
    Calculate cosine similarity between two embeddings.
    Measures the angle between two vector, Score = 1.0 is same meaning; Score = 0.0 is unrelated
    """
    sim = cosine_similarity(embedding1.reshape(1, -1), embedding2.reshape(1, -1))
    return float(sim[0][0])
    
def embeddingMatcher(g1, g2, threshold=0.85,
                     candidates=None):
    """
    Match classes and properties using sentence embeddings.
    For large ontologies, uses index candidates to limit
    comparison space.
    
    Parameters:
        g1, g2     : rdflib graphs
        threshold  : similarity threshold
        candidates : dict of index candidates (optional)
                     Used for large ontologies to avoid
                     O(n²) comparisons
    """
    from sentence_transformers import SentenceTransformer
    import numpy as np

    model = SentenceTransformer('all-MiniLM-L6-v2')

    classes1   = getClasses(g1)
    classes2   = getClasses(g2)
    obj_props1 = list(getObjectProperties(g1))
    obj_props2 = list(getObjectProperties(g2))

    total = len(classes1) * len(classes2)
    use_candidates = (candidates is not None and total > 1000000)

    if use_candidates:
        print(f"  Large ontology: using index candidates")
        cand_count = sum(len(v) for v in candidates.values())
        print(f"  ({total:,} pairs → "f"{cand_count:,} candidates)")
    
    #  Class Embeddings 
    labels1 = [getLabel(g1, c) for c in classes1]
    labels2 = [getLabel(g2, c) for c in classes2]

    emb1 = model.encode(labels1, show_progress_bar=False)
    emb2 = model.encode(labels2, show_progress_bar=False)

    label_to_emb2 = {str(classes2[i]): emb2[i]
        for i in range(len(classes2))}

    class_mappings = []
    MAX_EMB_CANDIDATES = 100000  
    processed_emb = 0
    emb_skipped = False
    
    for idx1, c1 in enumerate(classes1):

        # Early stopping only for large ontology
        if use_candidates:
            if processed_emb > MAX_EMB_CANDIDATES:
                if not emb_skipped:
                    print(f" !!Embedding candidate limit "
                      f"reached ({MAX_EMB_CANDIDATES:,})")
                    emb_skipped = True
                break
                
        if candidates is not None and c1 in candidates:
                processed_emb += len(candidates[c1])

        label1 = labels1[idx1]
        best_score = 0.0
        best_match = None

        if use_candidates and candidates is not None and c1 in candidates:
            compare_set = candidates[c1]
        else:
            compare_set = classes2

        processed_emb += len(compare_set)

        for c2 in compare_set:
            if str(c2) not in label_to_emb2:
                continue

            emb_c2 = label_to_emb2[str(c2)]
            norm1  = np.linalg.norm(emb1[idx1])
            norm2  = np.linalg.norm(emb_c2)
            
            if norm1 == 0 or norm2 == 0:
                continue

            score = float(np.dot(emb1[idx1], emb_c2) /
                (norm1 * norm2)
            )

            if score >= threshold and score > best_score:
                best_score = score
                best_match = c2

        if best_match is not None:
            class_mappings.append((c1, best_match, best_score))

    print(f"  Class mappings: {len(class_mappings)}")

    # Property Embeddings
    obj_mappings = []

    if obj_props1 and obj_props2:
        prop_labels1 = [getLabel(g1, p) for p in obj_props1]
        prop_labels2 = [getLabel(g2, p) for p in obj_props2]

        prop_emb1 = model.encode(prop_labels1, show_progress_bar=False)
        prop_emb2 = model.encode(prop_labels2, show_progress_bar=False)

        for i, p1 in enumerate(obj_props1):
            best_score = 0.0
            best_match = None

            for j, p2 in enumerate(obj_props2):
                norm1 = np.linalg.norm(prop_emb1[i])
                norm2 = np.linalg.norm(prop_emb2[j])

                if norm1 == 0 or norm2 == 0:
                    continue

                score = float(
                    np.dot(prop_emb1[i], prop_emb2[j]) /
                    (norm1 * norm2)
                )

                if score >= threshold and score > best_score:
                    best_score = score
                    best_match = p2

            if best_match is not None:
                obj_mappings.append((p1, best_match, best_score))

        print(f"  Obj prop mappings: {len(obj_mappings)}")

    return class_mappings, obj_mappings, []

Loading embedding model


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

 Model loaded


In [43]:
# Step2: Run Embedding matching on cmt.owl vs ekaw.owl

In [44]:
start_time = time.time()

emb_class, emb_obj, emb_data = embeddingMatcher(g1, g2, threshold=0.85)

elapsed_emb = round(time.time() - start_time, 2)
print(f"\nComputation time: {elapsed_emb}s")

# Display Class mappings
print(f"\n=== Embedding Class Mappings ===")
print(f"{'Source':<30} {'Target':<30} {'Score'}")
print("-" * 65)
for src, tgt, score in emb_class:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<30} {tgt_label:<30} {score:.3f}")

# Display property mappings
print(f"\n=== Embedding Property Mappings ===")
print(f"{'Source':<30} {'Target':<30} {'Score'}")
print("-" * 65)
for src, tgt, score in emb_obj:
    src_label = getLabel(g1, src)
    tgt_label = getLabel(g2, tgt)
    print(f"{src_label:<30} {tgt_label:<30} {score:.3f}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 5
  Obj prop mappings: 1

Computation time: 2.55s

=== Embedding Class Mappings ===
Source                         Target                         Score
-----------------------------------------------------------------
Person                         Person                         1.000
Document                       Document                       1.000
Conference                     Conference                     1.000
Review                         Review                         1.000
Paper                          Paper                          1.000

=== Embedding Property Mappings ===
Source                         Target                         Score
-----------------------------------------------------------------
writtenBy                      writtenBy                      1.000


In [45]:
# Try different thresholds for embedding matcher
for threshold in [0.85, 0.80, 0.75, 0.70]:
    print(f"\nThreshold: {threshold}")
    emb_c, emb_o, _ = embeddingMatcher(g1, g2, threshold=threshold)
    print(f"  Class mappings:    {len(emb_c)}")
    print(f"  Property mappings: {len(emb_o)}")
    print(f"  Total:             {len(emb_c) + len(emb_o)}")


Threshold: 0.85


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 5
  Obj prop mappings: 1
  Class mappings:    5
  Property mappings: 1
  Total:             6

Threshold: 0.8


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 6
  Obj prop mappings: 1
  Class mappings:    6
  Property mappings: 1
  Total:             7

Threshold: 0.75


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 7
  Obj prop mappings: 1
  Class mappings:    7
  Property mappings: 1
  Total:             8

Threshold: 0.7


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 10
  Obj prop mappings: 1
  Class mappings:    10
  Property mappings: 1
  Total:             11


In [46]:
# Step 3: Evaluate: Embedding with different thresholds to test the accuracy
for threshold in [0.85, 0.80, 0.75, 0.70]:
    # Run matcher
    emb_c, emb_o, emb_d = embeddingMatcher(g1, g2, threshold=threshold)
    
    # Save mappings
    emb_output = os.path.join(OA1_DIR,f"mloa-cmt-ekaw-embedding-{threshold}.ttl")
    saveMappings(emb_c, emb_o, emb_d, emb_output, g1=g1, g2=g2)
    
    # Evaluate
    print(f"\n=== Threshold {threshold} ===")
    p, r, f = evaluate(reference_file, emb_output)
    print(f"Classes: {len(emb_c)}, Props: {len(emb_o)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 5
  Obj prop mappings: 1
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-embedding-0.85.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        0
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  6

=== Threshold 0.85 ===
Evaluating: mloa-cmt-ekaw-embedding-0.85.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 1
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.833
Recall:    0.455
F-score:   0.588
Classes: 5, Props: 1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 6
  Obj prop mappings: 1
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-embedding-0.8.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        1
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  7

=== Threshold 0.8 ===
Evaluating: mloa-cmt-ekaw-embedding-0.8.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 2
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.714
Recall:    0.455
F-score:   0.556
Classes: 6, Props: 1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 7
  Obj prop mappings: 1
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-embedding-0.75.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        2
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  8

=== Threshold 0.75 ===
Evaluating: mloa-cmt-ekaw-embedding-0.75.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 3
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.625
Recall:    0.455
F-score:   0.526
Classes: 7, Props: 1


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 10
  Obj prop mappings: 1
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-embedding-0.7.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        5
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  11

=== Threshold 0.7 ===
Evaluating: mloa-cmt-ekaw-embedding-0.7.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 6
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.455
Recall:    0.455
F-score:   0.455
Classes: 10, Props: 1


In [47]:
# Step 4: Evaluate the embedding matcher (threshold=0.85)

# Reference alignment
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

# System alignment
emb_file = os.path.join(OA1_DIR, "mloa-cmt-ekaw-embedding-0.85.ttl")

# Run evaluation
precision, recall, f_score = evaluate(reference_file, emb_file)

Evaluating: mloa-cmt-ekaw-embedding-0.85.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 1
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.833
Recall:    0.455
F-score:   0.588


In [48]:
# Scetion 5: Results Tracking (Embedding Matching result)
results["Task"].append("cmt-ekaw")
results["Technique"].append("Embedding (threshold=0.85)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(elapsed_emb, 3))

# Display table
df_results = pd.DataFrame(results)
print(f"=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")

=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.130
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.290
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.037
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    2.550


In [49]:
# From the above result, we can observe that the precision of the result in embedding is higher than the previous methods.
# That means there are few incorrect mappings (low false positives) but the value of recall is keeping low. 
# To understand why, we perform a false negative analysis by comparing our system output against the reference alignment. 
# This reveals which correct mappings our system missed and why.

from rdflib import Graph

# load reference alignment
ref = Graph()
ref.parse(reference_file, format="turtle")

emb_best = Graph()
emb_best.parse(os.path.join(OA1_DIR, "mloa-cmt-ekaw-embedding-0.85.ttl"),format="turtle")

print("=== Mappings in Reference but Not in Embedding ===")
print("False Negatives\n")
for s, p, o in ref:
    if (s, p, o) not in emb_best and (o, p, s) not in emb_best:
        s_label = str(s).split("#")[-1].split("/")[-1]
        o_label = str(o).split("#")[-1].split("/")[-1]
        print(f"  {s_label:<30} → {o_label}")
print()

print("=== Mappings in Embedding but Not in Reference ===")
print("False Positives\n")

for s, p, o in emb_best:
    if (s, p, o) not in ref and (o, p, s) not in ref:
        s_label = str(s).split("#")[-1].split("/")[-1]
        o_label = str(o).split("#")[-1].split("/")[-1]
        print(f"  {s_label:<30} → {o_label}")

=== Mappings in Reference but Not in Embedding ===
False Negatives

  hasBeenAssigned                → reviewerOfPaper
  Author                         → Paper_Author
  assignedTo                     → hasReviewer
  ConferenceMember               → Conference_Participant
  PaperFullVersion               → Regular_Paper
  writtenBy                      → reviewWrittenBy

=== Mappings in Embedding but Not in Reference ===
False Positives

  writtenBy                      → writtenBy


#### Advanced technique 2: LLM-based Matcher

In [50]:
## Step 1: LLM as Oracle for Ontology Alignment
## Sources: Large Language Models as Oracles for Ontology Alignment by Sviatoslav Lushnei1 et al.
## Repository:https://arxiv.org/pdf/2508.08500

In [51]:
## Access API: openrouter.key 
import os

def loadAPIKey():
    """
    Load OpenRouter API key from local file.
    File is stored outside the notebook directory for security.
    """
    key_path = os.path.join(os.path.expanduser("~/Desktop/MLOA"),"files", "openrouter.key.txt")
    
    print(f"Looking for key at:")
    print(f"  {key_path}")
    print()
    
    if os.path.exists(key_path):
        with open(key_path, 'r') as f:
            key = f.read().strip()
        
        if key:
            print(f" API key loaded successfully!")
            masked = key[:8] + "..." + key[-4:]
            print(f"  Key preview: {masked}")
            return key
        else:
            raise ValueError(
                "Key file is empty!\n"
                "Please put your API key inside the file."
            )
    else:
        raise FileNotFoundError(
            f"Key file not found at:\n"
            f"  {key_path}\n"
            f"Please check the file location."
        )
try:
    api_key = loadAPIKey()
except Exception as e:
    print(f" The key is missing, Error: {e}")

Looking for key at:
  C:\Users\siuju/Desktop/MLOA\files\openrouter.key.txt

 API key loaded successfully!
  Key preview: sk-or-v1...0224


In [52]:
# Step2: Create LLM function with API
# Model: nvidia/nemotron-3-super-120b-a12b:free
# Code resue and make reference to LogMapLLM 
def askLLM(label1, label2, context="conference management", api_key=None):
    """
    Ask LLM whether two ontology labels represent the same concept.
    Parameters:
        label1  : string label from ontology 1
        label2  : string label from ontology 2
        context : domain context for better understanding
        api_key : OpenRouter API key
    Returns:
        True  = LLM thinks they are equivalent
        False = LLM thinks they are Not equivalent
    """
# Check the exact match before asking LLM
    if label1.lower() == label2.lower():
        return True

    prompt = f"""Task: Ontology alignment for {context} systems.

    Are "{label1}" and "{label2}" the same concept?

    Example of same concepts:
    - "Paper" and "Article" → yes (both mean academic paper)
    - "Author" and "Writer" → yes (both mean person who writes)

    Example of different concepts:
    - "Reviewer" and "Review" → no (person vs document)
    - "Conference" and "Paper" → no (event vs document)
    
    Answer with only 'yes' or 'no'."""

    # Call API
    headers = {"Authorization": f"Bearer {api_key}","Content-Type": "application/json"}

    body = {
        "model": "nvidia/nemotron-3-super-120b-a12b:free",
        "messages": [
            {
                "role": "user",
                "content": prompt
            }
        ],
        "max_tokens": 10,    
        "temperature": 0.0   
    }

    try:
        response = requests.post("https://openrouter.ai/api/v1/chat/completions", headers=headers,json=body,timeout=30)

        if response.status_code == 200:
            result = response.json()
            answer = result["choices"][0]["message"][
                "content"].strip().lower()
            if "yes" in answer:
                return True
            else:
                return False

        elif response.status_code == 429:
            print(f" !! Rate limit, waiting 10s...")
            time.sleep(5)
            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=body,
                timeout=30
            )
            if response.status_code == 200:
                result = response.json()
                answer = result["choices"][0]["message"]["content"].strip().lower()
                return "yes" in answer
            return False

        else:
            print(f"API Error: {response.status_code}")
            return False

    except Exception as e:
        print(f"Request Error: {e}")
        return False
        
def llmMatcher(g1, g2, api_key, context="conference management", embedding_threshold=0.6, max_candidates=50, candidates=None):
    """
    LLM-based ontology matcher.

    Strategy:
    1. Use embedding to find candidates (low threshold)
    2. Ask LLM to verify each candidate
    3. Return verified mappings
    """
    #print("=== LLM Matcher ===")
    # print(f"Step 1: Finding candidates with "
          #f"the lowest embedding threshold={embedding_threshold}")

    emb_candidates_c, emb_candidates_o, _ = embeddingMatcher(g1, g2,threshold=embedding_threshold, candidates=candidates)

    all_candidates = emb_candidates_c + emb_candidates_o
    print(f"Candidates found: {len(all_candidates)}")

    # Control the amount to LLm to save cost 
    if len(all_candidates) > max_candidates:
        print(f"Limiting to top {max_candidates} candidates")
        # Sorted by the embedding score 
        all_candidates = sorted(all_candidates, key=lambda x: x[2],reverse=True)[:max_candidates]

    # print(f"\nStep 2: Asking LLM to verify {len(all_candidates)} candidates")
    print(f"Domain context: '{context}'\n")

    verified_mappings = []
    rejected_mappings = []

    for i, (e1, e2, emb_score) in enumerate(all_candidates):

        label1 = getLabel(g1, e1)
        label2 = getLabel(g2, e2)

        is_equivalent = askLLM(label1, label2, context=context, api_key=api_key)

        if is_equivalent:
            verified_mappings.append((e1, e2, emb_score))
            status = "YES"
        else:
            rejected_mappings.append((e1, e2, emb_score))
            status = "NO"

        print(f"  [{i+1}/{len(all_candidates)}] "
              f"{label1:<25} → {label2:<25} {status}")

        time.sleep(0.5)

    # print(f"\nStep 3: Results")
    # print(f"  Verified:  {len(verified_mappings)}")
    # print(f"  Rejected:  {len(rejected_mappings)}")

    return verified_mappings

In [53]:
# Step3: Run LLM Matcher on cmt.owl vs ekaw.owl

start_time = time.time()

llm_mappings = llmMatcher(g1, g2, api_key=api_key, context="conference management and paper review system", embedding_threshold=0.6, max_candidates=50)

elapsed_llm = round(time.time() - start_time, 2)
print(f"\nTotal time: {elapsed_llm}s")

llm_class_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.Class) in g1]
llm_obj_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.ObjectProperty) in g1]

print(f"\n=== Mapping Summary ===")
print(f"Class mappings:    {len(class_mappings)}")
print(f"Object prop:       {len(obj_mappings)}")
print(f"Data prop:         {len(data_mappings)}")
print(f"Total:             "
      f"{len(class_mappings) + len(obj_mappings) + len(data_mappings)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 14
  Obj prop mappings: 14
Candidates found: 28
Domain context: 'conference management and paper review system'

  [1/28] Meta-Reviewer             → Possible_Reviewer         NO
  [2/28] Reviewer                  → Review                    YES
  [3/28] Person                    → Person                    YES
  [4/28] Document                  → Document                  YES
  [5/28] Conference                → Conference                YES
  [6/28] ConferenceChair           → Conference                YES
  [7/28] ConferenceMember          → Conference                YES
  [8/28] Review                    → Review                    YES
  [9/28] Paper                     → Paper                     YES
  [10/28] Author                    → Paper_Author              NO
  [11/28] Co-author                 → Paper_Author              NO
  [12/28] PaperAbstract             → Paper                     YES
  [13/28] PaperFullVersion          → Paper                     N

In [54]:
# Step 4: Evaluate: LLM Matcher

llm_class_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.Class) in g1]
llm_obj_mappings = [(s, t, sc) for s, t, sc in llm_mappings if (s, RDF.type, OWL.ObjectProperty) in g1]

print(f"LLM Class mappings:    {len(llm_class_mappings)}")
print(f"LLM Property mappings: {len(llm_obj_mappings)}")

llm_output = os.path.join(OA1_DIR, "mloa-cmt-ekaw-llm.ttl")
saveMappings(llm_class_mappings, llm_obj_mappings, [], llm_output, g1=g1, g2=g2)

print()

reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")
precision, recall, f_score = evaluate(reference_file, llm_output)

LLM Class mappings:    10
LLM Property mappings: 2
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-llm.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        5
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     1
  owl:sameAs:             0
  Total:                  12

Evaluating: mloa-cmt-ekaw-llm.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 7
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.417
Recall:    0.455
F-score:   0.435


In [55]:
#  Step 5:Results Tracking: Add LLM Matcher result
results["Task"].append("cmt-ekaw")
results["Technique"].append("LLM (NVIDIA Nemotron)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(92.44)

# Display table
df_results = pd.DataFrame(results)
print(f"\n=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")


=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.130
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.290
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.037
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    2.550
cmt-ekaw     LLM (NVIDIA Nemotron)                    0.417    0.455    0.435   92.440


#### Advanced technique 3: Merge Embedding and LLM-based Matcher

In [56]:
# Step 1:
print("=== Merging Embedding + LLM Mappings ===")

# Combine class mappings
final_class = mergeMapping(
    emb_class, # Embedding class mappings
    llm_class_mappings # LLM class mappings
)

# Combine property mappings
final_obj = mergeMapping(
    emb_obj, # Embedding property mappings
    llm_obj_mappings # LLM property mappings
)

print(f"\nEmbedding class mappings:  {len(emb_class)}")
print(f"LLM class mappings:        {len(llm_class_mappings)}")
print(f"Merged class mappings:     {len(final_class)}")
print(f"\nEmbedding obj mappings:    {len(emb_obj)}")
print(f"LLM obj mappings:          {len(llm_obj_mappings)}")
print(f"Merged obj mappings:       {len(final_obj)}")

# 
final_output = os.path.join(OA1_DIR, "mloa-cmt-ekaw-final.ttl")
saveMappings(final_class, final_obj, [], final_output, g1=g1, g2=g2)
print(f"\n Embedding + LLM alignment saved!")

=== Merging Embedding + LLM Mappings ===

Embedding class mappings:  5
LLM class mappings:        10
Merged class mappings:     10

Embedding obj mappings:    1
LLM obj mappings:          2
Merged obj mappings:       2
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-final.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        5
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     1
  owl:sameAs:             0
  Total:                  12

 Embedding + LLM alignment saved!


In [57]:
# Step 2: Evaluate the system of Embedding + LLM alignment
reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")

precision, recall, f_score = evaluate(reference_file, final_output)

Evaluating: mloa-cmt-ekaw-final.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 7
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.417
Recall:    0.455
F-score:   0.435


In [58]:
# Step 3:Results Tracking (Embedding + LLM alignmen)

results["Task"].append("cmt-ekaw")
results["Technique"].append("Embedding + LLM")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(1.34 + 92.44, 2))

# Display table
df_results = pd.DataFrame(results)
print(f"\n=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")


=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.130
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.290
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.037
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    2.550
cmt-ekaw     LLM (NVIDIA Nemotron)                    0.417    0.455    0.435   92.440
cmt-ekaw     Embedding + LLM                          0.417    0.455    0.435   93.780


In [59]:
# Step 4: Check the difference between Embedding and LLM mapping 

print("=== Embedding Class Mappings ===")
for s, t, sc in emb_class:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

print(f"\n=== LLM Class Mappings ===")
for s, t, sc in llm_class_mappings:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

print(f"\n=== Merged Class Mappings ===")
for s, t, sc in final_class:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

=== Embedding Class Mappings ===
  Person                         → Person                         1.000
  Document                       → Document                       1.000
  Conference                     → Conference                     1.000
  Review                         → Review                         1.000
  Paper                          → Paper                          1.000

=== LLM Class Mappings ===
  Reviewer                       → Review                         0.810
  Person                         → Person                         1.000
  Document                       → Document                       1.000
  Conference                     → Conference                     1.000
  ConferenceChair                → Conference                     0.659
  ConferenceMember               → Conference                     0.740
  Review                         → Review                         1.000
  Paper                          → Paper                          1.000
  P

In [60]:
# Step 5:Filter LLM mapping with embedding score >=0.85

LLM_SCORE_THRESHOLD = 0.85

# Filter low score 
filtered_llm_class = [
    (s, t, sc) for s, t, sc in llm_class_mappings
    if sc >= LLM_SCORE_THRESHOLD
]

filtered_llm_obj = [
    (s, t, sc) for s, t, sc in llm_obj_mappings
    if sc >= LLM_SCORE_THRESHOLD
]

print(f"=== LLM Mappings Filtering ===")
print(f"Original LLM class mappings:  {len(llm_class_mappings)}")
print(f"Filtered LLM class mappings:  {len(filtered_llm_class)}")
print(f"\nKept mappings:")
for s, t, sc in filtered_llm_class:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

print(f"\nRemoved mappings:")
removed = [
    (s, t, sc) for s, t, sc in llm_class_mappings
    if sc < LLM_SCORE_THRESHOLD
]
for s, t, sc in removed:
    s_label = getLabel(g1, s)
    t_label = getLabel(g2, t)
    print(f"  {s_label:<30} → {t_label:<30} {sc:.3f}")

=== LLM Mappings Filtering ===
Original LLM class mappings:  10
Filtered LLM class mappings:  5

Kept mappings:
  Person                         → Person                         1.000
  Document                       → Document                       1.000
  Conference                     → Conference                     1.000
  Review                         → Review                         1.000
  Paper                          → Paper                          1.000

Removed mappings:
  Reviewer                       → Review                         0.810
  ConferenceChair                → Conference                     0.659
  ConferenceMember               → Conference                     0.740
  PaperAbstract                  → Paper                          0.756
  Meta-Review                    → Review                         0.700


In [61]:
# Step 6: Evaluate Embedding + Filtered LLM (Final)

# Merge Embedding + Filtered LLM
final_class_filtered = mergeMapping(emb_class, filtered_llm_class)
final_obj_filtered = mergeMapping(emb_obj, filtered_llm_obj)

print(f"Final class mappings:  {len(final_class_filtered)}")
print(f"Final obj mappings:    {len(final_obj_filtered)}")


final_filtered_output = os.path.join(OA1_DIR, "mloa-cmt-ekaw-final-filtered.ttl")
saveMappings(final_class_filtered, final_obj_filtered, [], final_filtered_output, g1=g1,g2=g2)

print()


reference_file = os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl")
precision, recall, f_score = evaluate(reference_file, final_filtered_output)

Final class mappings:  5
Final obj mappings:    1
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-final-filtered.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        0
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  6

Evaluating: mloa-cmt-ekaw-final-filtered.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 1
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.833
Recall:    0.455
F-score:   0.588


In [62]:
# Step 7: Results Tracking 
results["Task"].append("cmt-ekaw")
results["Technique"].append("Embedding + LLM Filtered (thd=0.85)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(1.38 + 92.44, 2))

df_results = pd.DataFrame(results)
print(f"\n=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")


=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.130
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.290
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.037
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    2.550
cmt-ekaw     LLM (NVIDIA Nemotron)                    0.417    0.455    0.435   92.440
cmt-ekaw     Embedding + LLM                          0.417    0.455    0.435   93.780
cmt-ekaw     Embedding + LLM Filtered (thd=0.85)      0.833    0.455    0.588   93.820


### Subtask OA.1 Completed MLOA system

In [63]:
# The system includes all the mapping
# • owl:equivalentClass (class mappings score >= 0.9)
# • rdfs:subClassOf (class mappings score < 0.9)
# • owl:equivalentProperty (property mappings >=0.9)
# • rdfs:subPropertyOf (property mappings <0.9)
# • owl:sameAs (instance mappings)

In [64]:
# Step 1: Instance Matching 
def instanceMatcher(g1, g2, threshold=0.9):
    """
    Mathces individuals between two ontologies and uses inverted index + early stopping for scalability.
    """
    individuals1 = list(getIndividuals(g1))
    individuals2 = list(getIndividuals(g2))

    # Limit individuals for scalability
    MAX_INDIVIDUALS = 10000
    if len(individuals1) > MAX_INDIVIDUALS:
        print(f"!! Limiting source: "
          f"{len(individuals1):,} → {MAX_INDIVIDUALS:,}")
        individuals1 = individuals1[:MAX_INDIVIDUALS]

    if len(individuals2) > MAX_INDIVIDUALS:
        print(f"!! Limiting target: "
          f"{len(individuals2):,} → {MAX_INDIVIDUALS:,}")
        individuals2 = individuals2[:MAX_INDIVIDUALS]

    print(f"=== Instance Matcher ===")
    print(f"Source individuals: {len(individuals1)}")
    print(f"Target individuals: {len(individuals2)}")

    if not individuals1 or not individuals2:
        print("  No individuals to match in one or both ontologies")
        print("  Skipping instance matching")
        return []

    # Build Index for scalability 
    print(f"  Building index")
    index = {}
    for ind2 in individuals2:
        label2 = getLabel(g2, ind2)
        tokens = tokenise(label2)
        for token in tokens:
            if token not in index:
                index[token] = set()
            index[token].add(ind2)

    print(f"  Index built: {len(index)} tokens")

    # Match using Index candidates 
    MAX_INST = 50000  # Early stopping limit
    mappings = []
    processed = 0 
    
    for i1 in individuals1:
        # Early stopping
        if processed >= MAX_INST:
            print(f" !! Instance limit reached "
                  f"({MAX_INST:,})")
            break

        label1 = getLabel(g1, i1)
        tokens1   = tokenise(label1)

        # Index candidates
        candidates_set = set()
        for token in tokens1:
            if token in index:
                candidates_set.update(index[token])

        if not candidates_set:
            processed += 1
            continue
            
        best_score = 0.0
        best_match = None

        for i2 in candidates_set:
            label2 = getLabel(g2, i2)
            score = lexicalSimilarity(label1, label2)

            if score >= threshold and score > best_score:
                best_score = score
                best_match = i2

        if best_match is not None:
            mappings.append((i1, best_match, best_score))

        processed += 1

    print(f"  Processed: {processed:,} individuals")
    print(f"Instance mappings found: {len(mappings)}")

    if mappings:
        print(f"\nSample mappings:")
        print(f"{'Source':<30} {'Target':<30} {'Score'}")
        print("-" * 65)
        for i1, i2, score in mappings[:5]:
            l1 = getLabel(g1, i1)
            l2 = getLabel(g2, i2)
            print(f"  {l1:<28} {l2:<28} {score:.3f}")
    return mappings

In [65]:
# Step 2: Run instance matcher on cmt.owl vs ekaw.owl
ind_mappings = instanceMatcher(g1, g2, threshold=0.9)

=== Instance Matcher ===
Source individuals: 0
Target individuals: 0
  No individuals to match in one or both ontologies
  Skipping instance matching


In [66]:
# Step 3: Save Complete Final Alignment

final_v2_output = os.path.join(OA1_DIR, "mloa-cmt-ekaw-final-v2.ttl")

saveMappings(final_class_filtered, final_obj_filtered, [], final_v2_output, individual_mappings=ind_mappings,g1=g1, g2=g2)

 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.1\mloa-cmt-ekaw-final-v2.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        0
  owl:equivalentProperty: 1
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  6


In [67]:
# Sction 4: Results Tracking of the Final MLOA System 
results["Task"].append("cmt-ekaw")
results["Technique"].append("Final MLOA System (All predicates)")
results["Precision"].append(round(precision, 3))
results["Recall"].append(round(recall, 3))
results["F-score"].append(round(f_score, 3))
results["Time(s)"].append(round(1.38 + 92.44, 2))

df_results = pd.DataFrame(results)
print(f"\n=== MLOA Results Tracking ===")
print(f"{'Task':<12} {'Technique':<35} {'Precision':>10}"
      f" {'Recall':>8} {'F-score':>8} {'Time(s)':>8}")
print("-" * 85)
for _, row in df_results.iterrows():
    print(f"{row['Task']:<12} {row['Technique']:<35}"
          f" {row['Precision']:>10.3f}"
          f" {row['Recall']:>8.3f}"
          f" {row['F-score']:>8.3f}"
          f" {row['Time(s)']:>8.3f}")


=== MLOA Results Tracking ===
Task         Technique                            Precision   Recall  F-score  Time(s)
-------------------------------------------------------------------------------------
cmt-ekaw     Lexical (threshold=0.9/0.8)              0.333    0.455    0.385    0.130
cmt-ekaw     Lexical + Structural (merged)            0.250    0.455    0.323    0.290
cmt-ekaw     Indexed Matcher                          0.312    0.455    0.370    0.037
cmt-ekaw     Embedding (threshold=0.85)               0.833    0.455    0.588    2.550
cmt-ekaw     LLM (NVIDIA Nemotron)                    0.417    0.455    0.435   92.440
cmt-ekaw     Embedding + LLM                          0.417    0.455    0.435   93.780
cmt-ekaw     Embedding + LLM Filtered (thd=0.85)      0.833    0.455    0.588   93.820
cmt-ekaw     Final MLOA System (All predicates)       0.833    0.455    0.588   93.820


### MLOA System 

In [68]:
# Step 1: Run all alighment techniques in sequence:
# 1. load ontologies
# 2. Inverted Index (scalability)
# 3. Lexical Matching 
# 4. Structural Matching (combined with Lexical)
# 5. Inverted Index (scalability)
# 6. Embedding Matching + LLM (advanced)
# 7. Instance Matching (owl:sameAs)
# 8. SaVE Alignment

def runMLOA(onto1_path, onto2_path, output_path, context="general domain", time_limit=7200, use_llm=False, api_key=None, g1=None, g2=None):
    """
    Run the complete MLOA alignment system.

    Parameters:
        onto1_path : path to source ontology (.owl)
        onto2_path : path to target ontology (.owl)
        output_path: path to save alignment (.ttl)
        context    : domain description for LLM
        time_limit : maximum seconds (default 2 hours)
        use_llm    : whether to use LLM matching
        api_key    : OpenRouter API key (for LLM)

    Returns:
        dict with results:
        {
            "precision": float,
            "recall":    float,
            "f_score":   float,
            "time":      float,
            "status":    "completed" or "timeout"
        }
        """
    onto1_name = os.path.basename(onto1_path).replace(".owl", "")
    onto2_name = os.path.basename(onto2_path).replace(".owl", "")

    print(f"\n{'='*55}")
    print(f"MLOA: {onto1_name} vs {onto2_name}")
    print(f"{'='*55}")

    start_total = time.time()

    try:
        # Step 1: Load Ontologies 
        print("Step 1: Loading ontologies")
        if g1 is None:
            g1 = loadOntology(onto1_path)
        if g2 is None:
            g2 = loadOntology(onto2_path)
        print(f"  Source: {len(getClasses(g1))} classes, "
              f"{len(list(getObjectProperties(g1)))} props")
        print(f"  Target: {len(getClasses(g2))} classes, "
              f"{len(list(getObjectProperties(g2)))} props")

        if time.time() - start_total > time_limit:
            return {"status": "timeout",
                    "time": time_limit}

        # Step 2: Build Inverted Index 
        print("\nStep 2: Building Inverted Index")
        index2 = buildIndex(g2)

        all_e1 = (list(getClasses(g1)) + list(getObjectProperties(g1)) + list(getDataProperties(g1)))
        all_e2 = (list(getClasses(g2)) + list(getObjectProperties(g2)) + list(getDataProperties(g2)))

        candidates, total_candidates = getCandidates(g1, g2, index2)
        brute_force = len(all_e1) * len(all_e2)

        print(f"  Index built: {len(index2)} tokens")
        print(f"  Brute force: {brute_force:,} comparisons")
        print(f"  With index:  {total_candidates:,} comparisons")
        if brute_force > 0:
            reduction = (1 - total_candidates / brute_force) * 100
            print(f"  Reduction:   {reduction:.1f}%")

        if time.time() - start_total > time_limit:
            return {"status": "timeout",
                    "time": time_limit}

        # Step 3: Lexical Matching (with Index)
        print("\nStep 3: Lexical Matching (using index)")
        
        lex_class = []
        lex_obj   = []
        lex_data  = []

        MAX_CANDIDATES = 500000  # 

        processed = 0
        skipped   = False

        for entity1, entity2_set in candidates.items():
            # Early stopping for very large ontologies
            if processed > MAX_CANDIDATES:
                if not skipped:
                    print(f"  Candidate limit reached "
                          f"({MAX_CANDIDATES:,}), stopping early")
                    skipped = True
                break

            
            label1     = getLabel(g1, entity1)
            best_score = 0.0
            best_match = None

            for entity2 in entity2_set:
                label2 = getLabel(g2, entity2)
                score  = lexicalSimilarity(label1, label2)

                # Class threshold = 0.9
                # Property threshold = 0.8
                is_class = (
                    entity1, RDF.type, OWL.Class
                ) in g1
                threshold = 0.9 if is_class else 0.9

                if score >= threshold and score > best_score:
                    best_score = score
                    best_match = entity2

            if best_match is not None:
                if (entity1, RDF.type, OWL.Class) in g1:
                    lex_class.append((entity1, best_match, best_score))
                elif (entity1, RDF.type,
                          OWL.ObjectProperty) in g1:
                    lex_obj.append((entity1, best_match, best_score))
                elif (entity1, RDF.type,
                          OWL.DatatypeProperty) in g1:
                    lex_data.append((entity1, best_match, best_score))

            processed += len(entity2_set)

        print(f"  Class: {len(lex_class)}, "
              f"Obj prop: {len(lex_obj)}, "
              f"Data prop: {len(lex_data)}")

        if time.time() - start_total > time_limit:
            return {"status": "timeout",
                    "time": time_limit}

        # Step 4: Structural Matching 
        print("\nStep 4: Structural Matching")
        try:
            # Skip if ontologies too large (>1M comparisons)
            total_classes = (len(getClasses(g1)) * len(getClasses(g2)))
            if total_classes > 1000000:
                print(f"!! Skipped (too large: "
                      f"{total_classes:,} comparisons)")
                merged_class = lex_class
            else:
                struct_class = structuralMatcher(g1, g2)
                merged_class = mergeMapping(
                    lex_class, struct_class
                )
            print(f"  Merged: {len(merged_class)} classes")
        except Exception as e:
            print(f"  !! Structural matching failed: {e}")
            print(f"  Using lexical mappings only...")
            merged_class = lex_class

        if time.time() - start_total > time_limit:
            return {"status": "timeout",
                    "time": time_limit}

        # Step 5: Embedding Matching 
        print("\nStep 5: Embedding Matching")
        emb_class, emb_obj, _ = embeddingMatcher(g1, g2, threshold=0.85, candidates=candidates)
        final_class = mergeMapping(merged_class, emb_class)
        final_obj   = mergeMapping(lex_obj, emb_obj)
        print(f"  Final class: {len(final_class)}, "
              f"Obj prop: {len(final_obj)}")

        if time.time() - start_total > time_limit:
            return {"status": "timeout",
                    "time": time_limit}

        # Step 6: LLM Matching (optional) 
        if use_llm and api_key:
            print("\nStep 6: LLM Matching")
            llm_mappings = llmMatcher(g1, g2,api_key=api_key,context=context,embedding_threshold=0.8,max_candidates=15,candidates=candidates )
            llm_class = [
                (s, t, sc) for s, t, sc in llm_mappings
                if (s, RDF.type, OWL.Class) in g1
                and sc >= 0.85
            ]
            llm_obj = [
                (s, t, sc) for s, t, sc in llm_mappings
                if (s, RDF.type, OWL.ObjectProperty) in g1
            ]
            final_class = mergeMapping(final_class, llm_class)
            final_obj = mergeMapping(final_obj, llm_obj)
            print(f"  After LLM: {len(final_class)} classes")
        else:
            print("\nStep 6: LLM Matching skipped")

        if time.time() - start_total > time_limit:
            return {"status": "timeout",
                    "time": time_limit}

        # Step 7: Instance Matching 
        print("\nStep 7: Instance Matching")
        ind_mappings = instanceMatcher(g1, g2, threshold=0.9)

        # Step 8: Save Alignment 
        print("\nStep 8: Saving alignment")
        saveMappings(final_class, final_obj, lex_data, output_path, individual_mappings=ind_mappings, g1=g1, g2=g2)

        elapsed = round(time.time() - start_total, 2)
        print(f"\n Completed in {elapsed}s")

        return {
            "status":  "completed",
            "time":    elapsed,
            "n_class": len(final_class),
            "n_obj":   len(final_obj),
            "n_inst":  len(ind_mappings)
        }

    except Exception as e:
        elapsed = round(time.time() - start_total, 2)
        print(f" Error: {e}")
        return {
            "status": "error",
            "error":  str(e),
            "time":   elapsed
        }

In [69]:
# Step 2:Run complete MLOA on cmt vs ekaw (Test)
test_output = os.path.join(output_dir, "mloa-cmt-ekaw-oa4.ttl")

test_output = os.path.join(output_dir, "mloa-cmt-ekaw-oa4.ttl")

result = runMLOA(
    onto1_path=os.path.join(CONFERENCE_ONTO_PATH, "cmt.owl"),
    onto2_path=os.path.join(CONFERENCE_ONTO_PATH, "ekaw.owl"),
    output_path=test_output,
    context="conference management",
    use_llm=True,
    api_key=api_key
)

if result["status"] == "completed":
    print()
    precision, recall, f_score = evaluate(os.path.join(CONFERENCE_REF_PATH, "cmt-ekaw.ttl"),
        test_output
    )


MLOA: cmt vs ekaw
Step 1: Loading ontologies
  Source: 29 classes, 49 props
  Target: 73 classes, 33 props

Step 2: Building Inverted Index
  Index built: 85 tokens
  Brute force: 9,328 comparisons
  With index:  539 comparisons
  Reduction:   94.2%

Step 3: Lexical Matching (using index)
  Class: 9, Obj prop: 5, Data prop: 2

Step 4: Structural Matching
  Merged: 14 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 5
  Obj prop mappings: 1
  Final class: 14, Obj prop: 5

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 6
  Obj prop mappings: 1
Candidates found: 7
Domain context: 'conference management'

  [1/7] Reviewer                  → Review                    YES
  [2/7] Person                    → Person                    YES
  [3/7] Document                  → Document                  YES
  [4/7] Conference                → Conference                YES
  [5/7] Review                    → Review                    YES
  [6/7] Paper                     → Paper                     YES
  [7/7] writtenBy                 → writtenBy                 YES


  After LLM: 14 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 0
Target individuals: 0
  No individuals to match in one or both ontologies
  Skipping instance matching

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-cmt-ekaw-oa4.ttl
  owl:equivalentClass:    9
  rdfs:subClassOf:        5
  owl:equivalentProperty: 7
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  21

 Completed in 16.43s

Evaluating: mloa-cmt-ekaw-oa4.ttl
Reference:  cmt-ekaw.ttl
─────────────────────────────────────────────
True Positives  (TP): 5
False Positives (FP): 16
False Negatives (FN): 6
─────────────────────────────────────────────
Precision: 0.238
Recall:    0.455
F-score:   0.312


### Subtask OA.2 (Apply the ontology alignment system to the Part 1 Ontology Coursework)


In [70]:
# Apply MLOA alignment system
# Domain: Animal Classification
# Soure ontology: Matthew_Ontoloy.ttl
# Target ontology: Astha_Ontology.ttl

In [71]:
# Step 1:Load Ontologies
# Paths 
ONTO_PART1_PATH = os.path.normpath(os.path.join(MLOA_BASE, "Ontology_Part1_outcome", "Matthew_Ontology_Part1.ttl"))
ONTO_PARTNER_PATH = os.path.normpath(os.path.join(MLOA_BASE, "Ontology_Part1_outcome", "Astha_Ontology_Part1.ttl"))

# Output alignment file
OA2_OUTPUT = os.path.normpath(os.path.join(OA2_DIR, "mloa-M_animal-A_animal.ttl"))

# Create output directory if not exists
os.makedirs(OA2_DIR, exist_ok=True)

# Verify files exist
print("=== OA.2 Path Verification ===")
for label, path in [
    ("M_Ontology", ONTO_PART1_PATH),
    ("A_Ontology", ONTO_PARTNER_PATH),
    ("Output dir", OA2_DIR)
]:
    exists = os.path.exists(path)
    status = "OK" if exists else "NOT FOUND"
    print(f"  [{status}] {label}")
    print(f"         {path}")

# Load both ontologies
print("\n=== Loading Ontologies ===")
g_ml = loadOntology(ONTO_PART1_PATH)
g_al = loadOntology(ONTO_PARTNER_PATH)

print("\n--- M_Ontology ---")
getOntologyInfo(g_ml, "M_Animal_Ontology")

print("\n--- A_Ontology---")
getOntologyInfo(g_al, "A_Animal_Ontology")

=== OA.2 Path Verification ===
  [OK] M_Ontology
         C:\Users\siuju\Desktop\MLOA\Ontology_Part1_outcome\Matthew_Ontology_Part1.ttl
  [OK] A_Ontology
         C:\Users\siuju\Desktop\MLOA\Ontology_Part1_outcome\Astha_Ontology_Part1.ttl
  [OK] Output dir
         C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.2

=== Loading Ontologies ===

--- M_Ontology ---
=== M_Animal_Ontology Summary ===
Classes:            24
Object properties:  5
Data properties:    5
Individuals:        11

First 5 classes and their labels:
  URI:   http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/Amphibia
  Label: Amphibia

  URI:   http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/Animal
  Label: Animal

  URI:   http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/AquaticHabitat
  Label: AquaticHabitat

  URI:   http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/Artiodactyla
  Label: Artiodactyla

  URI:   http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/Aves
  Label: Aves


--- A_Ontology-

In [72]:
# Step 2: Run MLOA Alignment System
# Applying the full MLOA pipeline to Part 1 ontologies (animal)
# Source: runMLOA() defined in Subtask OA.1 

import time

print("=== Running MLOA on Part 1 Ontologies ===")
print(f"Source: M_Animal_Ontology (ml:)")
print(f"Target: A_Animal_Ontology   (al:)")
print(f"Output: {OA2_OUTPUT}\n")

oa2_start = time.time()

result = runMLOA(
    onto1_path  = ONTO_PART1_PATH,
    onto2_path  = ONTO_PARTNER_PATH,
    output_path = OA2_OUTPUT,
    context     = "animal biology, wildlife taxonomy, diet, habitat and conservation",
    time_limit  = 3600,    # 1 hour limit
    use_llm     = True,    # LLM (NVIDIA Nemotron)
    api_key     = api_key, 
    g1          = g_ml,    # Pass pre-loaded graphs to avoid reloading
    g2          = g_al
)

oa2_elapsed = round(time.time() - oa2_start, 2)

print(f"\n=== OA.2 Result ===")
print(f"Status : {result.get('status')}")
print(f"Time   : {oa2_elapsed}s")
if result.get('status') == 'completed':
    print(f"Classes   mapped : {result.get('n_class', 'N/A')}")
    print(f"Obj props mapped : {result.get('n_obj',   'N/A')}")
    print(f"Instances mapped : {result.get('n_inst',  'N/A')}")

=== Running MLOA on Part 1 Ontologies ===
Source: M_Animal_Ontology (ml:)
Target: A_Animal_Ontology   (al:)
Output: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.2\mloa-M_animal-A_animal.ttl


MLOA: Matthew_Ontology_Part1.ttl vs Astha_Ontology_Part1.ttl
Step 1: Loading ontologies
  Source: 24 classes, 5 props
  Target: 48 classes, 7 props

Step 2: Building Inverted Index
  Index built: 83 tokens
  Brute force: 2,482 comparisons
  With index:  112 comparisons
  Reduction:   95.5%

Step 3: Lexical Matching (using index)
  Class: 5, Obj prop: 2, Data prop: 4

Step 4: Structural Matching
  Merged: 5 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 6
  Obj prop mappings: 1
  Final class: 6, Obj prop: 2

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 6
  Obj prop mappings: 1
Candidates found: 7
Domain context: 'animal biology, wildlife taxonomy, diet, habitat and conservation'

  [1/7] Animal                    → Animal                    YES
  [2/7] Carnivora                 → Carnivore                 YES
  [3/7] CarnivoreDiet             → Carnivore                 NO
  [4/7] Habitat                   → Habitat                   YES
  [5/7] HerbivoreDiet             → Herbivore                 YES
  [6/7] OmnivoreDiet              → Omnivore                  NO
  [7/7] hasDiet                   → hasDiet                   YES
  After LLM: 6 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 11
Target individuals: 99
  Building index
  Index built: 86 tokens
  Processed: 11 individuals
Instance mappings found: 0

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.2\mloa-M_animal-A_animal.ttl
  owl:equivalentClass:    6
  rdfs:subClassOf: 

In [73]:
# Step 3: Inspect Alignment Results
from rdflib import Graph
from rdflib.namespace import OWL, RDFS, RDF

# namespace constants
ML = "http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/"
AL = "http://www.city.ac.uk/inm713-in3067/2026/astha_limbu/"

def shortLabel(uri):
    uri = str(uri)
    if ML in uri:
        return "ml:" + uri.replace(ML, "")
    elif AL in uri:
        return "al:" + uri.replace(AL, "")
    return uri.split("/")[-1].split("#")[-1]
  
# Load computed alignment
alignment_oa2 = Graph()
alignment_oa2.parse(OA2_OUTPUT, format="turtle")

print(f"=== OA.2 Alignment Inspection ===")
print(f"Total triples in alignment: {len(alignment_oa2)}\n")

# Categorise by predicate
equiv_class = [(s, o) for s, p, o in alignment_oa2 if p == OWL.equivalentClass]
sub_class   = [(s, o) for s, p, o in alignment_oa2 if p == RDFS.subClassOf]
equiv_prop  = [(s, o) for s, p, o in alignment_oa2 if p == OWL.equivalentProperty]
sub_prop    = [(s, o) for s, p, o in alignment_oa2 if p == RDFS.subPropertyOf]
same_as     = [(s, o) for s, p, o in alignment_oa2 if p == OWL.sameAs]

print(f"owl:equivalentClass    : {len(equiv_class)}")
print(f"rdfs:subClassOf        : {len(sub_class)}")
print(f"owl:equivalentProperty : {len(equiv_prop)}")
print(f"rdfs:subPropertyOf     : {len(sub_prop)}")
print(f"owl:sameAs             : {len(same_as)}")

print(f"\n--- owl:equivalentClass ({len(equiv_class)}) ---")
for s, o in sorted(equiv_class, key=lambda x: shortLabel(x[0])):
    print(f"  {shortLabel(s):<35} ≡  {shortLabel(o)}")

print(f"\n--- rdfs:subClassOf ({len(sub_class)}) ---")
for s, o in sorted(sub_class, key=lambda x: shortLabel(x[0])):
    print(f"  {shortLabel(s):<35} ⊑  {shortLabel(o)}")

print(f"\n--- owl:equivalentProperty ({len(equiv_prop)}) ---")
for s, o in sorted(equiv_prop, key=lambda x: shortLabel(x[0])):
    print(f"  {shortLabel(s):<35} ≡  {shortLabel(o)}")

print(f"\n--- rdfs:subPropertyOf ({len(sub_prop)}) ---")
for s, o in sorted(sub_prop, key=lambda x: shortLabel(x[0])):
    print(f"  {shortLabel(s):<35} ⊑  {shortLabel(o)}")

print(f"\n--- owl:sameAs ({len(same_as)}) ---")
for s, o in same_as:
    print(f"  {shortLabel(s):<35} =  {shortLabel(o)}")

=== OA.2 Alignment Inspection ===
Total triples in alignment: 12

owl:equivalentClass    : 6
rdfs:subClassOf        : 0
owl:equivalentProperty : 6
rdfs:subPropertyOf     : 0
owl:sameAs             : 0

--- owl:equivalentClass (6) ---
  ml:Animal                           ≡  al:Animal
  ml:Carnivora                        ≡  al:Carnivore
  ml:CarnivoreDiet                    ≡  al:Carnivore
  ml:Habitat                          ≡  al:Habitat
  ml:HerbivoreDiet                    ≡  al:Herbivore
  ml:OmnivoreDiet                     ≡  al:Omnivore

--- rdfs:subClassOf (0) ---

--- owl:equivalentProperty (6) ---
  ml:hasConservationStatus            ≡  al:hasConservationStatus
  ml:hasDiet                          ≡  al:hasDiet
  ml:hasHabitat                       ≡  al:Habitat
  ml:hasLIfespan                      ≡  al:hasLifespan
  ml:hasScientificName                ≡  al:hasScientificSpeciesName
  ml:hasTopSpeed                      ≡  al:hasSpeed

--- rdfs:subPropertyOf (0) ---

--

### Subtask OA.3 (Create and excute a SPARQL query )

In [74]:
# Subtask OA.3 - Step 1: Load Ontologies + Alignment + Reasoning
# Source: Lab 8- support code by Ernesto Jiménez-Ruiz - queryLocalRDFGraph_WorldCities.py
# Repository: https://github.com/city-knowledge-graphs/python-2026/tree/main/lab8

from rdflib import Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL
from rdflib.util import guess_format
import owlrl
import os

# Output directory
os.makedirs(OA3_DIR, exist_ok=True)

# File paths
M_ONTO_PATH = ONTO_PART1_PATH    # has instances
A_ONTO_PATH = ONTO_PARTNER_PATH  # has classes
ALIGN_PATH = OA2_OUTPUT         # OA.2 alignment result

# === Step 1: Load all three files into One graph ===
print("=== OA.3 Step 1: Loading all files ===")
g_oa3 = Graph()

g_oa3.parse(M_ONTO_PATH, format="turtle")
print(f"After M_Ontology : {len(g_oa3):,} triples")

g_oa3.parse(A_ONTO_PATH, format="turtle")
print(f"After A_Ontology : {len(g_oa3):,} triples")

g_oa3.parse(ALIGN_PATH, format="turtle")
print(f"After Alignment  : {len(g_oa3):,} triples")

# === Step 2: Perform OWL Reasoning ===
print("\n=== OA.3 Step 2: Performing OWL Reasoning ===")
print("Applying OWL 2 RL semantics")

owlrl.DeductiveClosure(owlrl.OWLRL_Semantics, axiomatic_triples=False, datatype_axioms=False).expand(g_oa3)

print(f"Triples after reasoning: {len(g_oa3):,}")
print("Reasoning complete!")

=== OA.3 Step 1: Loading all files ===
After M_Ontology : 200 triples
After A_Ontology : 1,163 triples
After Alignment  : 1,175 triples

=== OA.3 Step 2: Performing OWL Reasoning ===
Applying OWL 2 RL semantics
Triples after reasoning: 3,095
Reasoning complete!


In [75]:
# Step 2: Verify reasoning worked
# For example: Check that ml:Tiger is now inferred as al:Animal

# Define namespaces
ML = Namespace("http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/")
AL = Namespace("http://www.city.ac.uk/inm713-in3067/2026/astha_limbu/")

print("=== Reasoning Verification ===")
print("Checking if ml:Tiger is inferred as al:Animal\n")

# Check key inferred triples
checks = [
    (ML.Tiger,    RDF.type,  AL.Animal,  "ml:Tiger rdf:type al:Animal"),
    (ML.GreyWolf, RDF.type,  AL.Animal,  "ml:GreyWolf rdf:type al:Animal"),
    (ML.BaldEagle,RDF.type,  AL.Animal,  "ml:BaldEagle rdf:type al:Animal"),
    (ML.Tiger,    RDF.type,  ML.Animal,  "ml:Tiger rdf:type ml:Animal"),
]

for s, p, o, label in checks:
    exists = (s, p, o) in g_oa3
    status = "INFERRED" if exists else "NOT FOUND"
    print(f"  [{status}] {label}")

print("\nIf Inferred shows above = reasoning worked correctly!")
print("OA.3 SPARQL query will return results using both ontologies.")

=== Reasoning Verification ===
Checking if ml:Tiger is inferred as al:Animal

  [INFERRED] ml:Tiger rdf:type al:Animal
  [INFERRED] ml:GreyWolf rdf:type al:Animal
  [INFERRED] ml:BaldEagle rdf:type al:Animal
  [INFERRED] ml:Tiger rdf:type ml:Animal

If Inferred shows above = reasoning worked correctly!
OA.3 SPARQL query will return results using both ontologies.


In [76]:
# Step 3: SPARQL Query using both ontologies
# owl:equivalentClass and owl:equivalentProperty reasoning
# Requirement: at least 2 triple patterns, must return results
# Source: queryLocalRDFGraph_WorldCities.py by Ernesto Jiménez-Ruiz
# Repository: https://github.com/city-knowledge-graphs/python-2026/tree/main/lab8

import pandas as pd
import os

OA3_CSV_1 = os.path.normpath(os.path.join(OA3_DIR, "oa3_query1_results.csv"))
OA3_CSV_2 = os.path.normpath(os.path.join(OA3_DIR, "oa3_query2_results.csv"))
OA3_CSV_3 = os.path.normpath(os.path.join(OA3_DIR, "oa3_query3_results.csv"))
OA3_CSV_COMBINED = os.path.normpath(os.path.join(OA3_DIR, "oa3_combined_results.csv"))

# Helper: shorten URI to label only
def short(uri):
    if uri is None:
        return "N/A"
    return str(uri).split("/")[-1].split("#")[-1]

# Query 1: owl:equivalentClass reasoning
# al:Animal (class) + ml: properties
sparql_query_1 = """
PREFIX ml: <http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/>
PREFIX al: <http://www.city.ac.uk/inm713-in3067/2026/astha_limbu/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?animal ?scientificName ?diet ?conservationStatus
WHERE {
    ?animal rdf:type                 al:Animal .
    ?animal ml:hasScientificName     ?scientificName .
    ?animal ml:hasDiet               ?diet .
    ?animal ml:hasConservationStatus ?conservationStatus .
}
ORDER BY ?scientificName
"""

print("=" * 65)
print("Query 1: owl:equivalentClass reasoning")
print("=" * 65)
print("  1. ?animal rdf:type al:Animal          → al: class")
print("  2. ?animal ml:hasScientificName        → ml: property")
print("  3. ?animal ml:hasDiet                  → ml: property")
print("  4. ?animal ml:hasConservationStatus    → ml: property\n")

qres1 = g_oa3.query(sparql_query_1)
print(f"Results found: {len(qres1)}\n")
print(f"  {'Animal':<15} {'Scientific Name':<28} {'Diet':<18} {'Conservation'}")
print("  " + "-" * 80)

rows1 = []
for row in qres1:
    animal = short(row.animal)
    sci = str(row.scientificName)
    diet = short(row.diet)
    cons = str(row.conservationStatus)
    print(f"  {animal:<15} {sci:<28} {diet:<18} {cons}")
    rows1.append({
        "query"              : "Query1_equivalentClass",
        "animal"             : animal,
        "scientific_name"    : sci,
        "diet"               : diet,
        "conservation_status": cons
    })

df1 = pd.DataFrame(rows1)
df1.to_csv(OA3_CSV_1, index=False, encoding="utf-8")
print(f"\n  Saved to: {OA3_CSV_1}")

# Query 2: owl:equivalentProperty reasoning
# ml:Animal (class) + al: properties 
sparql_query_2 = """
PREFIX ml: <http://www.city.ac.uk/inm713-in3067/2026/Matthew_Lau/>
PREFIX al: <http://www.city.ac.uk/inm713-in3067/2026/astha_limbu/>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?animal ?diet ?conservationStatus
WHERE {
    ?animal rdf:type                 ml:Animal .
    ?animal al:hasDiet               ?diet .
    ?animal al:hasConservationStatus ?conservationStatus .
}
ORDER BY ?animal
"""

print("\n" + "=" * 65)
print("Query 2: owl:equivalentProperty reasoning")
print("=" * 65)
print("  1. ?animal rdf:type ml:Animal          → ml: class")
print("  2. ?animal al:hasDiet                  → al: property")
print("  3. ?animal al:hasConservationStatus    → al: property\n")

qres2 = g_oa3.query(sparql_query_2)
print(f"Results found: {len(qres2)}\n")
print(f"  {'Animal':<15} {'Diet':<25} {'Conservation'}")
print("  " + "-" * 60)

rows2 = []
for row in qres2:
    animal = short(row.animal)
    diet = short(row.diet)
    cons = str(row.conservationStatus)
    print(f"  {animal:<15} {diet:<25} {cons}")
    rows2.append({
        "query"              : "Query2_equivalentProperty",
        "animal"             : animal,
        "scientific_name"    : "N/A",
        "diet"               : diet,
        "conservation_status": cons
    })

df2 = pd.DataFrame(rows2)
df2.to_csv(OA3_CSV_2, index=False, encoding="utf-8")
print(f"\n  Saved to: {OA3_CSV_2}")

# Summary
print("\n" + "=" * 65)
print("OA.3 Summary")
print("=" * 65)
print(f"  Query 1 (equivalentClass)    : {len(qres1)} results")
print(f"  Query 2 (equivalentProperty) : {len(qres2)} results")

Query 1: owl:equivalentClass reasoning
  1. ?animal rdf:type al:Animal          → al: class
  2. ?animal ml:hasScientificName        → ml: property
  3. ?animal ml:hasDiet                  → ml: property
  4. ?animal ml:hasConservationStatus    → ml: property

Results found: 3

  Animal          Scientific Name              Diet               Conservation
  --------------------------------------------------------------------------------
  GreyWolf        Canis lupus                  Carnivore_diet     Least concern (LC)
  BaldEagle       Hailaeetus leucocephalus     Carnivore_diet     Least concern
  Tiger           Pathera tigris               Carnivore_diet     Endangered (EN)

  Saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.3\oa3_query1_results.csv

Query 2: owl:equivalentProperty reasoning
  1. ?animal rdf:type ml:Animal          → ml: class
  2. ?animal al:hasDiet                  → al: property
  3. ?animal al:hasConservationStatus    → al: property

Results found:

### Subtask OA.4 (Evaluate the ontology alignment systems with the OAEI tasks)

In [77]:
# Evaluate MLOA system on all OAEI datasets.
# Report Precision, Recall, F-score and computation time.
# Datasets:
"""
- Conference (21 tasks)
- Digital Humanities (8 tasks)
- Circular Economy (2 tasks)
- Anatomy (1 task)
- Bio-ML (5 tasks)
- Knowledge Graph (5 tasks)
"""
# Naming convention: mola-onto1-onto2.ttl

'\n- Conference (21 tasks)\n- Digital Humanities (8 tasks)\n- Circular Economy (2 tasks)\n- Anatomy (1 task)\n- Bio-ML (5 tasks)\n- Knowledge Graph (5 tasks)\n'

### Subtask OA.4.1 (The OAEI - Conference Track (21 tasks))

In [78]:
# Run MLOA on all 21 conference alignment tasks
import os
import pandas as pd

# Conference ontology pairs with reference alignments
conference_pairs = [
    ("cmt",        "conference"),
    ("cmt",        "confOf"),
    ("cmt",        "edas"),
    ("cmt",        "ekaw"),
    ("cmt",        "iasted"),
    ("cmt",        "sigkdd"),
    ("conference", "confOf"),
    ("conference", "edas"),
    ("conference", "ekaw"),
    ("conference", "iasted"),
    ("conference", "sigkdd"),
    ("confOf",     "edas"),
    ("confOf",     "ekaw"),
    ("confOf",     "iasted"),
    ("confOf",     "sigkdd"),
    ("edas",       "ekaw"),
    ("edas",       "iasted"),
    ("edas",       "sigkdd"),
    ("ekaw",       "iasted"),
    ("ekaw",       "sigkdd"),
    ("iasted",     "sigkdd"),
]

# Results storage
oa4_results = []

print(f"=== OA.4 Conference Track ===")
print(f"Total tasks: {len(conference_pairs)}")
print(f"Using LLM: Yes")
print(f"Time limit: 2 hours per task\n")

# Run each task
for i, (onto1, onto2) in enumerate(conference_pairs):

    print(f"\n[{i+1}/{len(conference_pairs)}] "
          f"{onto1} vs {onto2}")

    # File paths
    onto1_path = os.path.join(CONFERENCE_ONTO_PATH, f"{onto1}.owl")
    onto2_path = os.path.join(CONFERENCE_ONTO_PATH, f"{onto2}.owl")
    output_path = os.path.join(output_dir,f"mloa-{onto1}-{onto2}.ttl")
    reference_path = os.path.join(CONFERENCE_REF_PATH,f"{onto1}-{onto2}.ttl")

     # Check files exist
    if not os.path.exists(onto1_path):
        print(f"  Missing: {onto1}.owl")
        continue
    if not os.path.exists(onto2_path):
        print(f"  Missing: {onto2}.owl")
        continue

    # Run MLOA
    result = runMLOA(onto1_path=onto1_path, onto2_path=onto2_path, output_path=output_path,
        context="conference management",
        use_llm=True,
        api_key=api_key,
        time_limit=3600
    )

    # Evaluate if reference exists
    if result["status"] == "completed" and os.path.exists(reference_path):
        precision, recall, f_score = evaluate(reference_path, output_path )
        oa4_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": round(precision, 3),
            "Recall":    round(recall, 3),
            "F-score":   round(f_score, 3),
            "Time(s)":   result["time"],
            "Status":    "completed"
        })
        print(f"  P={precision:.3f} "
              f"R={recall:.3f} "
              f"F={f_score:.3f}")
    elif result["status"] == "completed":
        oa4_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   result["time"],
            "Status":    "no reference"
        })
    else:
        oa4_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   result.get("time", 0),
            "Status":    result["status"]
        })

# Display results table
print(f"\n{'='*70}")
print(f"=== Conference Track Results ===")
print(f"{'='*70}")

df_oa4 = pd.DataFrame(oa4_results)
print(f"{'Task':<25} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} "
      f"{'Time(s)':>8} {'Status'}")
print("-" * 70)

for _, row in df_oa4.iterrows():
    p = f"{row['Precision']:.3f}" \
        if row['Precision'] is not None else "N/A"
    r = f"{row['Recall']:.3f}" \
        if row['Recall'] is not None else "N/A"
    f = f"{row['F-score']:.3f}" \
        if row['F-score'] is not None else "N/A"
    print(f"{row['Task']:<25} {p:>10} "
          f"{r:>8} {f:>8} "
          f"{row['Time(s)']:>8.1f} {row['Status']}")

# Summary statistics
completed = df_oa4[df_oa4['Status'] == 'completed']
if len(completed) > 0:
    print(f"\n=== Summary ===")
    print(f"Completed tasks:  {len(completed)}/{len(conference_pairs)}")
    print(f"Average Precision: {completed['Precision'].mean():.3f}")
    print(f"Average Recall:    {completed['Recall'].mean():.3f}")
    print(f"Average F-score:   {completed['F-score'].mean():.3f}")
    print(f"Total time:        {df_oa4['Time(s)'].sum():.1f}s")

=== OA.4 Conference Track ===
Total tasks: 21
Using LLM: Yes
Time limit: 2 hours per task


[1/21] cmt vs conference

MLOA: cmt vs conference
Step 1: Loading ontologies
  Source: 29 classes, 49 props
  Target: 59 classes, 46 props

Step 2: Building Inverted Index
  Index built: 102 tokens
  Brute force: 10,824 comparisons
  With index:  705 comparisons
  Reduction:   93.5%

Step 3: Lexical Matching (using index)
  Class: 11, Obj prop: 5, Data prop: 1

Step 4: Structural Matching
  Merged: 13 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 5
  Obj prop mappings: 0
  Final class: 13, Obj prop: 5

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 6
  Obj prop mappings: 0
Candidates found: 6
Domain context: 'conference management'

  [1/6] Meta-Reviewer             → Reviewer                  YES
  [2/6] Reviewer                  → Reviewer                  YES
  [3/6] Person                    → Person                    YES
  [4/6] Conference                → Conference                YES
  [5/6] Review                    → Review                    YES
  [6/6] Paper                     → Paper                     YES


  After LLM: 13 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 0
Target individuals: 0
  No individuals to match in one or both ontologies
  Skipping instance matching

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-cmt-conference.ttl
  owl:equivalentClass:    11
  rdfs:subClassOf:        2
  owl:equivalentProperty: 6
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  19

 Completed in 9.47s
Evaluating: mloa-cmt-conference.ttl
Reference:  cmt-conference.ttl
─────────────────────────────────────────────
True Positives  (TP): 4
False Positives (FP): 15
False Negatives (FN): 11
─────────────────────────────────────────────
Precision: 0.211
Recall:    0.267
F-score:   0.235
  P=0.211 R=0.267 F=0.235

[2/21] cmt vs confOf

MLOA: cmt vs confOf
Step 1: Loading ontologies
  Source: 29 classes, 49 props
  Target: 38 classes, 13 props

Step 2: Building Inverted Index
  Index b

... (96 outputs left out)

  Class mappings: 13
  Obj prop mappings: 2
  Final class: 48, Obj prop: 2

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 18
  Obj prop mappings: 3
Candidates found: 21
Limiting to top 15 candidates
Domain context: 'conference management'

  [1/15] Person                    → Person                    YES
  [2/15] Deadline                  → Deadline                  YES
  [3/15] Place                     → Place                     YES
  [4/15] Author                    → Author                    YES
  [5/15] Registration_fee          → Registration_fee          YES
  [6/15] Speaker                   → Speaker                   YES
  [7/15] Main_office               → Main_office               YES
  [8/15] pay                       → pay                       YES
  [9/15] Document                  → Document                  YES
  [10/15] Listener                  → Listener                  YES
  [11/15] Conference_hall           → Conference_hall           YES
  [12/15] Fee                       → Fee                       YES
  [13/15] Review                    → Review             

  After LLM: 48 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 4
Target individuals: 0
  No individuals to match in one or both ontologies
  Skipping instance matching

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-iasted-sigkdd.ttl
  owl:equivalentClass:    35
  rdfs:subClassOf:        13
  owl:equivalentProperty: 2
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  50

 Completed in 20.21s
Evaluating: mloa-iasted-sigkdd.ttl
Reference:  iasted-sigkdd.ttl
─────────────────────────────────────────────
True Positives  (TP): 14
False Positives (FP): 36
False Negatives (FN): 1
─────────────────────────────────────────────
Precision: 0.280
Recall:    0.933
F-score:   0.431
  P=0.280 R=0.933 F=0.431

=== Conference Track Results ===
Task                       Precision   Recall  F-score  Time(s) Status
----------------------------------------------------------------------

### Subtask OA.4.2 (The OAEI - Digital Humanities Track (8  tasks))

In [79]:
# Self-contained cell with all required functions
import re

def loadOntologyOwlready(path):
    """
    Load ontology by fixing non-standard XSD literals.
    Required for .rdf files with date/int format issues.
    """
    with open(path, 'r',
              encoding='utf-8', errors='ignore') as f:
        content = f.read()

    # Fix xsd:date → xsd:string
    content = re.sub(
        r'rdf:datatype="http://www\.w3\.org/2001/'
        r'XMLSchema#date"',
        'rdf:datatype="http://www.w3.org/2001/'
        'XMLSchema#string"',
        content
    )
    # Fix xsd:int → xsd:string
    content = re.sub(
        r'rdf:datatype="http://www\.w3\.org/2001/'
        r'XMLSchema#int"',
        'rdf:datatype="http://www.w3.org/2001/'
        'XMLSchema#string"',
        content
    )
    # Fix xsd:integer → xsd:string
    content = re.sub(
        r'rdf:datatype="http://www\.w3\.org/2001/'
        r'XMLSchema#integer"',
        'rdf:datatype="http://www.w3.org/2001/'
        'XMLSchema#string"',
        content
    )
    # Fix xsd:float → xsd:string
    content = re.sub(
        r'rdf:datatype="http://www\.w3\.org/2001/'
        r'XMLSchema#float"',
        'rdf:datatype="http://www.w3.org/2001/'
        'XMLSchema#string"',
        content
    )
    # Fix short form &xsd;date
    content = re.sub(
        r'rdf:datatype="&xsd;date"',
        'rdf:datatype="http://www.w3.org/2001/'
        'XMLSchema#string"',
        content
    )
    content = re.sub(
        r'rdf:datatype="&xsd;int"',
        'rdf:datatype="http://www.w3.org/2001/'
        'XMLSchema#string"',
        content
    )

    g = Graph()
    try:
        g.parse(data=content, format="xml")
    except Exception as e:
        print(f" !! Parse failed: {e}")
    return g

def loadOntology(path):
    """
    Load ontology with fallback for .rdf files.
    All .rdf files use XSD fix version.
    """
    if path.endswith(".rdf"):
        return loadOntologyOwlready(path)

    g = Graph()
    try:
        if path.endswith(".ttl"):
            g.parse(path, format="turtle")
        else:
            g.parse(path, format="xml")
        return g
    except Exception as e:
        print(f" !! rdflib failed: {e}")
        return loadOntologyOwlready(path)

SKOS_CONCEPT = URIRef("http://www.w3.org/2004/02/skos/core#Concept")

def getClasses(g):
    """
    Extract all classes from an ontology graph.
    Supports OWL (owl:Class) and SKOS (skos:Concept).
    Returns whichever has more entities.
    """
    # OWL classes
    owl_classes = []
    for s in g.subjects(RDF.type, OWL.Class):
        if isinstance(s, URIRef):
            owl_classes.append(s)

    # SKOS concepts
    skos_classes = []
    for s in g.subjects(RDF.type, SKOS_CONCEPT):
        if isinstance(s, URIRef):
            skos_classes.append(s)

    if len(skos_classes) > len(owl_classes):
        return skos_classes
    else:
        return owl_classes

# Override getLabel() to support SKOS prefLabel
# Required for Digital Humanities ontologies
SKOS_PREFLABEL = URIRef("http://www.w3.org/2004/02/skos/core#prefLabel")

def getLabel(g, entity):
    """
    Extract label with SKOS support.
    Priority:
    1. skos:prefLabel (DH ontologies)
    2. rdfs:label (standard OWL)
    3. URI local name (fallback)
    """
    # skos:prefLabel 
    for label in g.objects(entity, SKOS_PREFLABEL):
        return str(label)

    # rdfs:label 
    for label in g.objects(entity, RDFS.label):
        return str(label)

    # URI local name 
    uri = str(entity)
    if "#" in uri:
        return uri.split("#")[-1]
    else:
        return uri.split("/")[-1]


# Paths
DH_ONTO_PATH = os.path.join(OAEI_BASE, "digital-humanities", "ontologies")
DH_REF_PATH = os.path.join(OAEI_BASE, "digital-humanities")

# 8 alignment tasks
dh_pairs = [
    ("defc",          "pactols"),
    ("dha",           "unesco"),
    ("idai",          "pactols"),
    ("idai",          "parthenos"),
    ("ironagedanube", "pactols"),
    ("oeai",          "parthenos"),
    ("pactols",       "parthenos"),
    ("tadirah",       "unesco"),
]

dh_results = []

print(f"=== OA.4 Digital Humanities Track ===")
print(f"Total tasks: {len(dh_pairs)}\n")

for i, (onto1, onto2) in enumerate(dh_pairs):

    print(f"\n[{i+1}/{len(dh_pairs)}] "
          f"{onto1} vs {onto2}")

    onto1_path = os.path.join(DH_ONTO_PATH, f"{onto1}.rdf")
    onto2_path = os.path.join(DH_ONTO_PATH, f"{onto2}.rdf")
    output_path = os.path.join(output_dir, f"mloa-{onto1}-{onto2}.ttl")
    reference_path = os.path.join(DH_REF_PATH, f"{onto1}-{onto2}.ttl")

    g1_loaded = loadOntology(onto1_path)
    g2_loaded = loadOntology(onto2_path)
    
    print(f"  g1: {len(getClasses(g1_loaded))} classes")
    print(f"  g2: {len(getClasses(g2_loaded))} classes")

    result = runMLOA(
        onto1_path=onto1_path,
        onto2_path=onto2_path,
        output_path=output_path,
        context="digital humanities and archaeology",
        use_llm=True,
        api_key=api_key,
        time_limit=7200,
        g1=g1_loaded,  
        g2=g2_loaded   
    )

    if result["status"] == "completed" and \
       os.path.exists(reference_path):
        precision, recall, f_score = evaluate(
            reference_path, output_path
        )
        dh_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": round(precision, 3),
            "Recall":    round(recall, 3),
            "F-score":   round(f_score, 3),
            "Time(s)":   result["time"],
            "Status":    "completed"
        })
        print(f"  P={precision:.3f} "
              f"R={recall:.3f} "
              f"F={f_score:.3f}")
    else:
        dh_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   result.get("time", 0),
            "Status":    result["status"]
        })
        print(f"  Status: {result['status']}")

# Display results
df_dh = pd.DataFrame(dh_results)
print(f"\n{'='*70}")
print(f"=== Digital Humanities Track Results ===")
print(f"{'='*70}")
print(f"{'Task':<30} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} "
      f"{'Time(s)':>8} {'Status'}")
print("-" * 70)

for _, row in df_dh.iterrows():
    p = f"{row['Precision']:.3f}" \
        if row['Precision'] is not None else "N/A"
    r = f"{row['Recall']:.3f}" \
        if row['Recall'] is not None else "N/A"
    f = f"{row['F-score']:.3f}" \
        if row['F-score'] is not None else "N/A"
    print(f"{row['Task']:<30} {p:>10} "
          f"{r:>8} {f:>8} "
          f"{row['Time(s)']:>8.1f} {row['Status']}")

completed = df_dh[df_dh['Status'] == 'completed']
if len(completed) > 0:
    print(f"\n=== Summary ===")
    print(f"Completed:     {len(completed)}/8")
    print(f"Avg Precision: "
          f"{completed['Precision'].mean():.3f}")
    print(f"Avg Recall:    "
          f"{completed['Recall'].mean():.3f}")
    print(f"Avg F-score:   "
          f"{completed['F-score'].mean():.3f}")
    print(f"Total time:    "
          f"{df_dh['Time(s)'].sum():.1f}s")

=== OA.4 Digital Humanities Track ===
Total tasks: 8


[1/8] defc vs pactols
  g1: 801 classes
  g2: 68 classes

MLOA: defc.rdf vs pactols.rdf
Step 1: Loading ontologies
  Source: 801 classes, 0 props
  Target: 68 classes, 0 props

Step 2: Building Inverted Index
  Index built: 97 tokens
  Brute force: 54,468 comparisons
  With index:  16 comparisons
  Reduction:   100.0%

Step 3: Lexical Matching (using index)
  Class: 0, Obj prop: 0, Data prop: 0

Step 4: Structural Matching
  Merged: 0 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 2
  Final class: 2, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 4
Candidates found: 4
Domain context: 'digital humanities and archaeology'

  [1/4] cave                      → cave                      YES
  [2/4] cairn                     → cairn                     YES
  [3/4] in cave                   → cave                      NO
  [4/4] pintadera                 → gruta pintada             NO


  After LLM: 2 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 0
Target individuals: 0
  No individuals to match in one or both ontologies
  Skipping instance matching

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-defc-pactols.ttl
  owl:equivalentClass:    2
  rdfs:subClassOf:        0
  owl:equivalentProperty: 0
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  2

 Completed in 18.01s
Evaluating: mloa-defc-pactols.ttl
Reference:  defc-pactols.ttl
─────────────────────────────────────────────
True Positives  (TP): 2
False Positives (FP): 0
False Negatives (FN): 8
─────────────────────────────────────────────
Precision: 1.000
Recall:    0.200
F-score:   0.333
  P=1.000 R=0.200 F=0.333

[2/8] dha vs unesco
  g1: 114 classes
  g2: 487 classes

MLOA: dha.rdf vs unesco.rdf
Step 1: Loading ontologies
  Source: 114 classes, 0 props
  Target: 487 classes, 0 props

Step 2: 

... (29 outputs left out)

  Class mappings: 8
  Final class: 8, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 15
Candidates found: 15
Domain context: 'digital humanities and archaeology'

  [1/15] Communicating             → Communication             YES
  [2/15] Acquisition               → Acquisitions              NO
  [3/15] Academic Publishing       → Publishing                YES
  [4/15] Social Networking         → Social media              YES
  [5/15] Drawing                   → Drawings                  NO
  [6/15] Audio Recording           → Sound recordings          NO
  [7/15] Cataloging                → Cataloguing               YES
  [8/15] Encoding                  → Encoding                  YES
  [9/15] Optical Character Recognition → Character recognition     YES
  [10/15] Photographing             → Photographs               NO
  [11/15] Digital Publishing        → Electronic publishing     YES
  [12/15] Information Retrieval     → Information retrieval     YES
  [13/15] Mapping                   → Maps                      NO
  [14/15] Searching           

  After LLM: 8 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 169
Target individuals: 10
  Building index
  Index built: 15 tokens
  Processed: 169 individuals
Instance mappings found: 0

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-tadirah-unesco.ttl
  owl:equivalentClass:    5
  rdfs:subClassOf:        3
  owl:equivalentProperty: 0
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  8

 Completed in 67.64s
Evaluating: mloa-tadirah-unesco.ttl
Reference:  tadirah-unesco.ttl
─────────────────────────────────────────────
True Positives  (TP): 7
False Positives (FP): 1
False Negatives (FN): 8
─────────────────────────────────────────────
Precision: 0.875
Recall:    0.467
F-score:   0.609
  P=0.875 R=0.467 F=0.609

=== Digital Humanities Track Results ===
Task                            Precision   Recall  F-score  Time(s) Status
----------------------------------------

### Subtask OA.4.3 (The OAEI - CircularEconomy Track (2  tasks))

In [80]:
# Paths
CE_ONTO_PATH = os.path.join(OAEI_BASE, "circular-economy", "ontologies")
CE_REF_PATH = os.path.join(OAEI_BASE, "circular-economy")

# 2 alignment tasks
ce_pairs = [
    ("CEON", "BiOnto"),
    ("CEON", "MATONTO"),
]

ce_results = []

print(f"=== OA.4 Circular Economy Track ===")
print(f"Total tasks: {len(ce_pairs)}\n")

for i, (onto1, onto2) in enumerate(ce_pairs):

    print(f"\n[{i+1}/{len(ce_pairs)}] "
          f"{onto1} vs {onto2}")

    onto1_path = os.path.join(CE_ONTO_PATH, f"{onto1}.rdf")
    onto2_path = os.path.join(CE_ONTO_PATH, f"{onto2}.rdf")
    output_path = os.path.join(output_dir,f"mloa-{onto1}-{onto2}.ttl")
    reference_path = os.path.join(CE_REF_PATH, f"{onto1}-{onto2}.ttl")

    result = runMLOA(
        onto1_path=onto1_path,
        onto2_path=onto2_path,
        output_path=output_path,
        context="circular economy and sustainability",
        use_llm=True,
        api_key=api_key,
        time_limit=7200
    )

    if result["status"] == "completed" and os.path.exists(reference_path):
        precision, recall, f_score = evaluate(reference_path, output_path)
        ce_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": round(precision, 3),
            "Recall":    round(recall, 3),
            "F-score":   round(f_score, 3),
            "Time(s)":   result["time"],
            "Status":    "completed"
        })
        print(f"  P={precision:.3f} "
              f"R={recall:.3f} "
              f"F={f_score:.3f}")
    else:
        ce_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   result.get("time", 0),
            "Status":    result["status"]
        })
        print(f"  Status: {result['status']}")

# Display results
df_ce = pd.DataFrame(ce_results)
print(f"\n{'='*70}")
print(f"=== Circular Economy Track Results ===")
print(f"{'='*70}")
print(f"{'Task':<25} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} "
      f"{'Time(s)':>8} {'Status'}")
print("-" * 70)

for _, row in df_ce.iterrows():
    p = f"{row['Precision']:.3f}" \
        if row['Precision'] is not None else "N/A"
    r = f"{row['Recall']:.3f}" \
        if row['Recall'] is not None else "N/A"
    f = f"{row['F-score']:.3f}" \
        if row['F-score'] is not None else "N/A"
    print(f"{row['Task']:<25} {p:>10} "
          f"{r:>8} {f:>8} "
          f"{row['Time(s)']:>8.1f} {row['Status']}")

completed = df_ce[df_ce['Status'] == 'completed']
if len(completed) > 0:
    print(f"\n=== Summary ===")
    print(f"Completed:     {len(completed)}/2")
    print(f"Avg Precision: "
          f"{completed['Precision'].mean():.3f}")
    print(f"Avg Recall:    "
          f"{completed['Recall'].mean():.3f}")
    print(f"Avg F-score:   "
          f"{completed['F-score'].mean():.3f}")
    print(f"Total time:    "
          f"{df_ce['Time(s)'].sum():.1f}s")

=== OA.4 Circular Economy Track ===
Total tasks: 2


[1/2] CEON vs BiOnto

MLOA: CEON.rdf vs BiOnto.rdf
Step 1: Loading ontologies
  Source: 228 classes, 126 props
  Target: 779 classes, 64 props

Step 2: Building Inverted Index
  Index built: 771 tokens
  Brute force: 338,352 comparisons
  With index:  3,359 comparisons
  Reduction:   99.0%

Step 3: Lexical Matching (using index)
  Class: 48, Obj prop: 6, Data prop: 6

Step 4: Structural Matching
  Merged: 50 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 27
  Obj prop mappings: 0
  Final class: 52, Obj prop: 6

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 31
  Obj prop mappings: 0
Candidates found: 31
Limiting to top 15 candidates
Domain context: 'circular economy and sustainability'

  [1/15] Person                    → Person                    YES
  [2/15] Biofuel                   → Biofuel                   YES
  [3/15] Coal                      → Coal                      YES
  [4/15] Energy                    → Energy                    YES
  [5/15] Nickel                    → Nickel                    YES
  [6/15] Information               → Information               YES
  [7/15] Matter                    → Matter                    YES
  [8/15] Quantity                  → Quantity                  YES
  [9/15] Participation             → Participation             YES
  [10/15] Biomass                   → Biomass                   YES
  [11/15] Petroleum                 → Petroleum                 YES
  [12/15] Cellulose                 → Cellulose                 YES
  [13/15] Iron                      → Iron 

  After LLM: 52 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 103
Target individuals: 0
  No individuals to match in one or both ontologies
  Skipping instance matching

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-CEON-BiOnto.ttl
  owl:equivalentClass:    50
  rdfs:subClassOf:        2
  owl:equivalentProperty: 12
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  64

 Completed in 68.18s
Evaluating: mloa-CEON-BiOnto.ttl
Reference:  CEON-BiOnto.ttl
─────────────────────────────────────────────
True Positives  (TP): 26
False Positives (FP): 38
False Negatives (FN): 3
─────────────────────────────────────────────
Precision: 0.406
Recall:    0.897
F-score:   0.559
  P=0.406 R=0.897 F=0.559

[2/2] CEON vs MATONTO

MLOA: CEON.rdf vs MATONTO.rdf
Step 1: Loading ontologies
  Source: 228 classes, 126 props
  Target: 847 classes, 82 props

Step 2: Building Inverted Index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 41
  Obj prop mappings: 3
  Final class: 49, Obj prop: 12

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Class mappings: 41
  Obj prop mappings: 4
Candidates found: 45
Limiting to top 15 candidates
Domain context: 'circular economy and sustainability'

  [1/15] Role                      → role                      YES
  [2/15] Energy                    → energy                    YES
  [3/15] Cerium                    → Cerium                    YES
  [4/15] Chromium                  → Chromium                  YES
  [5/15] Nickel                    → Nickel                    YES
  [6/15] Niobium                   → Niobium                   YES
  [7/15] Praseodymium              → Praseodymium              YES
  [8/15] Promethium                → Promethium                YES
  [9/15] Samarium                  → Samarium                  YES
  [10/15] Scandium                  → Scandium                  YES
  [11/15] Silicon                   → Silicon                   YES
  [12/15] Tantalum                  → Tantalum                  YES
  [13/15] Titanium                  → Titan

  After LLM: 49 classes

Step 7: Instance Matching
=== Instance Matcher ===
Source individuals: 103
Target individuals: 131
  Building index
  Index built: 74 tokens
  Processed: 103 individuals
Instance mappings found: 8

Sample mappings:
Source                         Target                         Score
-----------------------------------------------------------------
  Kilogram Per Liter           kilogram per cubic meter     0.905
  Centimeter                   centimeter                   1.000
  Gram Per Cubic Meter         cubic meter per kilogram     0.955
  Cubic Meter                  cubic meter                  1.000
  Ohm                          ohm                          1.000

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-CEON-MATONTO.ttl
  owl:equivalentClass:    46
  rdfs:subClassOf:        3
  owl:equivalentProperty: 15
  rdfs:subPropertyOf:     0
  owl:sameAs:             8
  Total:                  72

 C

### Subtask OA.4.4 (The OAEI -  Anatomy  Track (1  tasks))

In [81]:
# OA.4.4 - Anatomy Track (1 task)

# Paths
ANATOMY_ONTO_PATH = os.path.join(OAEI_BASE, "anatomy", "ontologies")
ANATOMY_REF_PATH = os.path.join(OAEI_BASE, "anatomy")

anatomy_results = []

print(f"=== OA.4 Anatomy Track ===")
print(f"Total tasks: 1")
print(f"Using LLM:   Yes (max 30 candidates)\n")

result = runMLOA(
    onto1_path=os.path.join(ANATOMY_ONTO_PATH, "human.owl"),
    onto2_path=os.path.join(ANATOMY_ONTO_PATH, "mouse.owl"),
    output_path=os.path.join(output_dir, "mloa-human-mouse.ttl"),
    context="human and mouse anatomy biology",
    use_llm=True,
    api_key=api_key,
    time_limit=7200
)

reference_path = os.path.join(ANATOMY_REF_PATH, "human-mouse.ttl")

if result["status"] == "completed" and os.path.exists(reference_path):
    precision, recall, f_score = evaluate(reference_path, os.path.join(output_dir, "mloa-human-mouse.ttl"))
    anatomy_results.append({
        "Task":      "human-mouse",
        "Precision": round(precision, 3),
        "Recall":    round(recall, 3),
        "F-score":   round(f_score, 3),
        "Time(s)":   result["time"],
        "Status":    "completed"
    })
else:
    anatomy_results.append({
        "Task":      "human-mouse",
        "Precision": None,
        "Recall":    None,
        "F-score":   None,
        "Time(s)":   result.get("time", 0),
        "Status":    result["status"]
    })

# Display results
df_anatomy = pd.DataFrame(anatomy_results)
print(f"\n{'='*70}")
print(f"=== Anatomy Track Results ===")
print(f"{'='*70}")
print(f"{'Task':<20} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} "
      f"{'Time(s)':>8} {'Status'}")
print("-" * 70)

for _, row in df_anatomy.iterrows():
    p = f"{row['Precision']:.3f}" \
        if row['Precision'] is not None else "N/A"
    r = f"{row['Recall']:.3f}" \
        if row['Recall'] is not None else "N/A"
    f = f"{row['F-score']:.3f}" \
        if row['F-score'] is not None else "N/A"
    print(f"{row['Task']:<20} {p:>10} "
          f"{r:>8} {f:>8} "
          f"{row['Time(s)']:>8.1f} {row['Status']}")

if len(df_anatomy) > 0:
    completed = df_anatomy[
        df_anatomy['Status'] == 'completed'
    ]
    print(f"\n=== Summary ===")
    print(f"Completed: {len(completed)}/1")
    if len(completed) > 0:
        print(f"Precision: "
              f"{completed['Precision'].mean():.3f}")
        print(f"Recall:    "
              f"{completed['Recall'].mean():.3f}")
        print(f"F-score:   "
              f"{completed['F-score'].mean():.3f}")
        print(f"Time:      "
              f"{df_anatomy['Time(s)'].sum():.1f}s")

=== OA.4 Anatomy Track ===
Total tasks: 1
Using LLM:   Yes (max 30 candidates)


MLOA: human vs mouse
Step 1: Loading ontologies
  Source: 3304 classes, 2 props
  Target: 2744 classes, 3 props

Step 2: Building Inverted Index
  Index built: 1280 tokens
  Brute force: 9,081,582 comparisons
  With index:  187,027 comparisons
  Reduction:   97.9%

Step 3: Lexical Matching (using index)
  Class: 1391, Obj prop: 2, Data prop: 0

Step 4: Structural Matching
!! Skipped (too large: 9,066,176 comparisons)
  Merged: 1391 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (9,066,176 pairs → 187,027 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 410
  Obj prop mappings: 2
  Final class: 1425, Obj prop: 2

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (9,066,176 pairs → 187,027 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 494
  Obj prop mappings: 2
Candidates found: 496
Limiting to top 15 candidates
Domain context: 'human and mouse anatomy biology'

  [1/15] Lip                       → lip                       YES
  [2/15] Nipple                    → nipple                    YES
  [3/15] Endometrium               → endometrium               YES
  [4/15] Choroid                   → choroid                   YES
  [5/15] Meninges                  → meninges                  YES
  [6/15] Ileum                     → ileum                     YES
  [7/15] Ureter                    → ureter                    YES
  [8/15] Larynx                    → larynx                    YES
  [9/15] Nasopharynx               → nasopharynx               YES
  [10/15] Pharynx                   → pharynx                   YES
  [11/15] Limb                      → limb          

 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-human-mouse.ttl
  owl:equivalentClass:    1405
  rdfs:subClassOf:        20
  owl:equivalentProperty: 2
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  1427

 Completed in 60.07s
Evaluating: mloa-human-mouse.ttl
Reference:  human-mouse.ttl
─────────────────────────────────────────────
True Positives  (TP): 1123
False Positives (FP): 304
False Negatives (FN): 393
─────────────────────────────────────────────
Precision: 0.787
Recall:    0.741
F-score:   0.763

=== Anatomy Track Results ===
Task                  Precision   Recall  F-score  Time(s) Status
----------------------------------------------------------------------
human-mouse               0.787    0.741    0.763     60.1 completed

=== Summary ===
Completed: 1/1
Precision: 0.787
Recall:    0.741
F-score:   0.763
Time:      60.1s


### Subtask OA.4.5 (The OAEI - Bio-ML  Track (5  tasks))

In [82]:
# Scalability notes:
# - Structural matching skipped (>1M comparisons)
# - LLM enabled with max 15 candidates per task
# - 2 hour time limit per task

SKOS_CONCEPT = URIRef("http://www.w3.org/2004/02/skos/core#Concept")

def getClasses(g):
    owl_classes = [
        s for s in g.subjects(RDF.type, OWL.Class)
        if isinstance(s, URIRef)
    ]
    skos_classes = [
        s for s in g.subjects(RDF.type, SKOS_CONCEPT)
        if isinstance(s, URIRef)
    ]
    if len(skos_classes) > len(owl_classes):
        return skos_classes
    else:
        return owl_classes

BIOML_ONTO_PATH = os.path.join(OAEI_BASE, "bio-ml", "ontologies")
BIOML_REF_PATH = os.path.join(OAEI_BASE, "bio-ml")

bioml_pairs = [
    ("ncit",           "doid",
     "ncit-doid"),
    ("omim",           "ordo",
     "omim-ordo"),
    ("snomed.body",    "fma.body",
     "snomed.body-fma.body"),
    ("snomed.neoplas", "ncit.neoplas",
     "snomed.neoplas-ncit.neoplas"),
    ("snomed.pharm",   "ncit.pharm",
     "snomed.pharm-ncit.pharm"),
]

bioml_results = []

print(f"=== OA.4 Bio-ML Track ===")
print(f"Total tasks: {len(bioml_pairs)}")
print(f"Techniques:  Lexical + Index + Embedding + LLM")
print(f"             (Structural skipped: scalability)")
print(f"             (LLM: max 15 candidates per task)")
print(f"Time limit:  2 hours per task\n")

for i, (onto1, onto2, ref_name) in enumerate(bioml_pairs):

    print(f"\n[{i+1}/{len(bioml_pairs)}] "
          f"{onto1} vs {onto2}")

    onto1_path = os.path.join(BIOML_ONTO_PATH, f"{onto1}.owl")
    onto2_path = os.path.join(BIOML_ONTO_PATH, f"{onto2}.owl")
    output_path = os.path.join(output_dir, f"mloa-{onto1}-{onto2}.ttl")
    reference_path = os.path.join(BIOML_REF_PATH, f"{ref_name}.ttl")

    if not os.path.exists(onto1_path):
        print(f" Missing: {onto1}.owl")
        bioml_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   0,
            "Status":    "missing file"
        })
        continue

    result = runMLOA(
        onto1_path=onto1_path,
        onto2_path=onto2_path,
        output_path=output_path,
        context="biomedical and clinical terminology",
        use_llm=True,
        api_key=api_key,
        time_limit=7200
    )

    if result["status"] == "timeout":
        print(f" !! Timeout after "
              f"{result['time']:.1f}s (>2hrs)")
        bioml_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   result["time"],
            "Status":    "timeout (>2hrs)"
        })
        continue

    if result["status"] == "completed" and os.path.exists(reference_path):
        precision, recall, f_score = evaluate(reference_path, output_path)
        bioml_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": round(precision, 3),
            "Recall":    round(recall, 3),
            "F-score":   round(f_score, 3),
            "Time(s)":   result["time"],
            "Status":    "completed"
        })
        print(f"  P={precision:.3f} "
              f"R={recall:.3f} "
              f"F={f_score:.3f} "
              f"Time={result['time']:.1f}s")
    else:
        bioml_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   result.get("time", 0),
            "Status":    result["status"]
        })
        print(f"  Status: {result['status']}")

# Display results
df_bioml = pd.DataFrame(bioml_results)
print(f"\n{'='*75}")
print(f"=== Bio-ML Track Results ===")
print(f"{'='*75}")
print(f"{'Task':<30} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} "
      f"{'Time(s)':>8} {'Status'}")
print("-" * 75)

for _, row in df_bioml.iterrows():
    p = f"{row['Precision']:.3f}" \
        if row['Precision'] is not None else "N/A"
    r = f"{row['Recall']:.3f}" \
        if row['Recall'] is not None else "N/A"
    f = f"{row['F-score']:.3f}" \
        if row['F-score'] is not None else "N/A"
    print(f"{row['Task']:<30} {p:>10} "
          f"{r:>8} {f:>8} "
          f"{row['Time(s)']:>8.1f} {row['Status']}")

completed = df_bioml[
    df_bioml['Status'] == 'completed'
]
timeout = df_bioml[
    df_bioml['Status'].str.contains(
        'timeout', na=False
    )
]

print(f"\n=== Summary ===")
print(f"Completed: {len(completed)}/5")
print(f"Timeout:   {len(timeout)}/5")
if len(completed) > 0:
    print(f"Avg Precision: "
          f"{completed['Precision'].mean():.3f}")
    print(f"Avg Recall:    "
          f"{completed['Recall'].mean():.3f}")
    print(f"Avg F-score:   "
          f"{completed['F-score'].mean():.3f}")
print(f"Total time: "
      f"{df_bioml['Time(s)'].sum():.1f}s")

=== OA.4 Bio-ML Track ===
Total tasks: 5
Techniques:  Lexical + Index + Embedding + LLM
             (Structural skipped: scalability)
             (LLM: max 15 candidates per task)
Time limit:  2 hours per task


[1/5] ncit vs doid

MLOA: ncit vs doid
Step 1: Loading ontologies
  Source: 15762 classes, 97 props
  Target: 8465 classes, 15 props

Step 2: Building Inverted Index
  Index built: 4658 tokens
  Brute force: 134,484,320 comparisons
  With index:  3,456,159 comparisons
  Reduction:   97.4%

Step 3: Lexical Matching (using index)
  Candidate limit reached (500,000), stopping early
  Class: 831, Obj prop: 0, Data prop: 0

Step 4: Structural Matching
!! Skipped (too large: 133,425,330 comparisons)
  Merged: 831 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (133,425,330 pairs → 3,456,159 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 92
  Obj prop mappings: 0
  Final class: 849, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (133,425,330 pairs → 3,456,159 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 121
  Obj prop mappings: 0
Candidates found: 121
Limiting to top 15 candidates
Domain context: 'biomedical and clinical terminology'

  [1/15] Myelomeningocele          → myelomeningocele          YES
  [2/15] Mitochondrial Myopathy    → mitochondrial myopathy    YES
  [3/15] Meningocele               → meningocele               YES
  [4/15] Seborrheic Dermatitis     → seborrheic dermatitis     YES
  [5/15] Neurodermatitis           → neurodermatitis           YES
  [6/15] Angular Cheilitis         → angular cheilitis         YES
  [7/15] Telogen Effluvium         → telogen effluvium         YES
  [8/15] Hypogonadotropic Hypogonadism → hypogonadotropic hypogonadism YES
  [9/15] Hepatorenal Syndrome      → hepatorenal syndrome      YES
  [10/15] Parotitis                 → parotitis                 YES
  [11/15] Endometrial Adenosquamous 

=== Instance Matcher ===
Source individuals: 0
Target individuals: 0
  No individuals to match in one or both ontologies
  Skipping instance matching

Step 8: Saving alignment
 Alignment saved to: C:\Users\siuju\Desktop\MLOA\Ontology_Alignments\OA.4\mloa-ncit-doid.ttl
  owl:equivalentClass:    837
  rdfs:subClassOf:        12
  owl:equivalentProperty: 0
  rdfs:subPropertyOf:     0
  owl:sameAs:             0
  Total:                  849

 Completed in 356.57s


... (17 outputs left out)

  Large ontology: using index candidates
  (653,012,000 pairs → 5,377,151 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 54
  Obj prop mappings: 0
  Final class: 904, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (653,012,000 pairs → 5,377,151 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 71
  Obj prop mappings: 0
Candidates found: 71
Limiting to top 15 candidates
Domain context: 'biomedical and clinical terminology'

  [1/15] Granisetron hydrochloride → Granisetron Hydrochloride YES
  [2/15] Bedaquiline fumarate      → Bedaquiline Fumarate      YES
  [3/15] Lonoctocog alfa           → Lonoctocog Alfa           YES
  [4/15] Fumaric acid              → Fumaric Acid              YES
  [5/15] Heptanoic acid            → Heptanoic Acid            YES
  [6/15] Polyunsaturated fatty acid → Polyunsaturated Fatty Acid YES
  [7/15] Saturated fatty acid      → Saturated Fatty Acid      YES
  [8/15] Phenylmercuric nitrate    → Phenylmercuric Nitrate    YES
  [9/15] Pseudoephedrine sulfate   → Pseudoephedrine Sulfate   YES
  [10/15] Antipsychotic agent       → Antipsychotic Agent       YES
  [11/15] Levomethadyl acetate hydrochloride

Evaluating: mloa-snomed.pharm-ncit.pharm.ttl
Reference:  snomed.pharm-ncit.pharm.ttl
─────────────────────────────────────────────
True Positives  (TP): 549
False Positives (FP): 355
False Negatives (FN): 5254
─────────────────────────────────────────────
Precision: 0.607
Recall:    0.095
F-score:   0.164
  P=0.607 R=0.095 F=0.164 Time=588.4s

=== Bio-ML Track Results ===
Task                            Precision   Recall  F-score  Time(s) Status
---------------------------------------------------------------------------
ncit-doid                           0.349    0.063    0.107    356.6 completed
omim-ordo                           0.561    0.053    0.097    587.1 completed
snomed.body-fma.body                0.087    0.000    0.001   1165.7 completed
snomed.neoplas-ncit.neoplas         0.150    0.005    0.009    718.2 completed
snomed.pharm-ncit.pharm             0.607    0.095    0.164    588.4 completed

=== Summary ===
Completed: 5/5
Timeout:   0/5
Avg Precision: 0.351
Avg Recall

### Subtask OA.4.6 (The OAEI - Knowledge Graph Track (5 tasks))

In [83]:
# Step 1: Reasonable size pairs
# memoryalpha(349MB) + memorybeta(338MB)
# memoryalpha(349MB) + stexpanded(76MB)
# Time limit: 30 minutes per task
import warnings
import logging
from rdflib import Graph
from rdflib.util import guess_format

warnings.filterwarnings("ignore")
logging.getLogger("rdflib").setLevel(logging.ERROR)

SKOS_CONCEPT = URIRef("http://www.w3.org/2004/02/skos/core#Concept")

def getClasses(g):
    owl_classes = [
        s for s in g.subjects(RDF.type, OWL.Class)
        if isinstance(s, URIRef)
    ]
    skos_classes = [
        s for s in g.subjects(RDF.type, SKOS_CONCEPT)
        if isinstance(s, URIRef)
    ]
    if len(skos_classes) > len(owl_classes):
        return skos_classes
    else:
        return owl_classes
        
def getIndividuals(g):
    """
    Extract all individuals from an ontology graph.
    Supports both owl:NamedIndividual and direct rdf:type assertions.
    Filters out images and system namespaces.
    """
    system_namespaces = {
        "http://www.w3.org/2002/07/owl#",
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "http://www.w3.org/2000/01/rdf-schema#",
        "http://www.w3.org/2001/XMLSchema#",
        "http://www.w3.org/2004/02/skos/core#",
    }
    image_extensions = {
        '.jpg', '.jpeg', '.png', '.gif',
        '.svg', '.webp', '.pdf', '.mp3',
        '.mp4', '.wav', '.ogg'
    }

    individuals = set()

    # Method 1：owl:NamedIndividual
    for s in g.subjects(RDF.type, OWL.NamedIndividual):
        if isinstance(s, URIRef):
            individuals.add(s)

    # Method 2：rdf:type 
    classes = set(getClasses(g))
    for s, p, o in g.triples(
        (None, RDF.type, None)
    ):
        if not isinstance(s, URIRef):
            continue
        if not isinstance(o, URIRef):
            continue
        o_str = str(o)
        if any(o_str.startswith(ns)
               for ns in system_namespaces):
            continue
        s_str = str(s).lower()
        if any(ext in s_str
               for ext in image_extensions):
            continue
        if s in classes:
            continue
        individuals.add(s)

    return list(individuals)

KG_ONTO_PATH = os.path.join(OAEI_BASE, "knowledge-graph", "ontologies")
KG_REF_PATH = os.path.join(OAEI_BASE, "knowledge-graph")

kg_pairs_reasonable = [
    ("memoryalpha", "memorybeta",
     "memoryalpha-memorybeta"),
    ("memoryalpha", "stexpanded",
     "memoryalpha-stexpanded"),
]

def loadOntologyKG(path):
    """
    Load large KG ontology with error handling for non-standard XSD datatypes (e.g. gMonthDay).
    Source: Lab Session Digital Humanities fix, Ernesto Jiménez-Ruiz
    """
    g = Graph()
    try:
        # Suppress datatype warnings for large KG files
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            g.parse(path, format=guess_format(path))
        print(f"  Loaded: {len(g):,} triples")
        
    except Exception as e:
        print(f"  Warning: {e}")
        print(f"  Retrying with xml format...")
        g.parse(path, format="xml")
        print(f"  Loaded: {len(g):,} triples")
    return g
    
kg_results = []

print(f"=== OA.4 KG Track - Reasonable Pairs ===")
print(f"Total tasks: {len(kg_pairs_reasonable)}")
print(f"Techniques:  Lexical + Index + Embedding + LLM")
print(f"             (Structural skipped: scalability)")
print(f"             (LLM: max 15 candidates per task)")
print(f"Time limit:  30 minutes per task\n")

for i, (onto1, onto2, ref_name) in enumerate(kg_pairs_reasonable):

    task_start = time.time()

    print(f"\n[{i+1}/{len(kg_pairs_reasonable)}] "
          f"{onto1} vs {onto2}")

    onto1_path = os.path.join(KG_ONTO_PATH, f"{onto1}.owl")
    onto2_path = os.path.join(KG_ONTO_PATH, f"{onto2}.owl")
    output_path = os.path.join(output_dir,f"mloa-{onto1}-{onto2}.ttl")
    reference_path = os.path.join(KG_REF_PATH, f"{ref_name}.ttl")

    size1 = os.path.getsize(onto1_path) / (1024*1024)
    size2 = os.path.getsize(onto2_path) / (1024*1024)
    print(f"  File sizes: {size1:.1f}MB "
          f"+ {size2:.1f}MB = "
          f"{size1+size2:.1f}MB total")

    # Pre-load with KG-safe loader 
    print(f"  Loading ontologies (KG-safe mode)")
    try:
        g1 = loadOntologyKG(onto1_path)
        g2 = loadOntologyKG(onto2_path)
        
    except Exception as e:
        print(f"  !! Loading failed: {e}")
        kg_results.append({
            "Task": f"{onto1}-{onto2}",
            "Precision": None, "Recall": None,
            "F-score": None,
            "Time(s)": round(time.time()-task_start, 1),
            "Status": "load_error"
        })
        continue
        
    # Pass pre-loaded graphs to runMLOA
    result = runMLOA(
        onto1_path=onto1_path,
        onto2_path=onto2_path,
        output_path=output_path,
        context="knowledge graph alignment",
        use_llm=True,
        api_key=api_key,
        time_limit=1800,
        g1 =g1,
        g2 =g2,
    )

    task_elapsed = round(time.time() - task_start, 1)
    print(f"  Task elapsed: {task_elapsed}s "
          f"({task_elapsed/60:.1f} mins)")

    if result["status"] == "timeout":
        print(f" !! Timeout after {task_elapsed:.1f}s")
        kg_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   task_elapsed,
            "Status":    "timeout (>30min)"
        })
        continue

    if result["status"] == "completed" and os.path.exists(reference_path):
        precision, recall, f_score = evaluate(
            reference_path, output_path
        )
        kg_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": round(precision, 3),
            "Recall":    round(recall, 3),
            "F-score":   round(f_score, 3),
            "Time(s)":   task_elapsed,
            "Status":    "completed"
        })
        print(f"  P={precision:.3f} "
              f"R={recall:.3f} "
              f"F={f_score:.3f}")
    else:
        kg_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   task_elapsed,
            "Status":    result.get("status", "error")
        })
        print(f"  Status: {result.get('status')}")

# Display Step 1: Reasonable size pairs results
df_kg = pd.DataFrame(kg_results)
print(f"\n{'='*75}")
print(f"=== KG Track - Reasonable Pairs Results ===")
print(f"{'='*75}")
print(f"{'Task':<35} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} "
      f"{'Time(s)':>8} {'Status'}")
print("-" * 75)

for _, row in df_kg.iterrows():
    p = f"{row['Precision']:.3f}" \
        if row['Precision'] is not None else "N/A"
    r = f"{row['Recall']:.3f}" \
        if row['Recall'] is not None else "N/A"
    f = f"{row['F-score']:.3f}" \
        if row['F-score'] is not None else "N/A"
    print(f"{row['Task']:<35} {p:>10} "
          f"{r:>8} {f:>8} "
          f"{row['Time(s)']:>8.1f} {row['Status']}")

=== OA.4 KG Track - Reasonable Pairs ===
Total tasks: 2
Techniques:  Lexical + Index + Embedding + LLM
             (Structural skipped: scalability)
             (LLM: max 15 candidates per task)
Time limit:  30 minutes per task


[1/2] memoryalpha vs memorybeta
  File sizes: 349.9MB + 338.4MB = 688.4MB total
  Loading ontologies (KG-safe mode)
  Loaded: 2,526,928 triples
  Loaded: 2,542,909 triples

MLOA: memoryalpha vs memorybeta
Step 1: Loading ontologies
  Source: 3625 classes, 0 props
  Target: 5714 classes, 0 props

Step 2: Building Inverted Index
  Index built: 3397 tokens
  Brute force: 20,713,250 comparisons
  With index:  2,845,575 comparisons
  Reduction:   86.3%

Step 3: Lexical Matching (using index)
  Candidate limit reached (500,000), stopping early
  Class: 0, Obj prop: 0, Data prop: 0

Step 4: Structural Matching
!! Skipped (too large: 20,713,250 comparisons)
  Merged: 0 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (20,713,250 pairs → 2,845,575 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 18
  Final class: 18, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (20,713,250 pairs → 2,845,575 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 22
Candidates found: 22
Limiting to top 15 candidates
Domain context: 'knowledge graph alignment'

  [1/15] Skorr                     → Skorr                     YES
  [2/15] Fictional starships       → Fictional starships       YES
  [3/15] Ktarians                  → Ktarians                  YES
  [4/15] Ceremonies                → Ceremonies                YES
  [5/15] Elaysians                 → Elaysians                 YES
  [6/15] Maneuvers                 → Maneuvers                 YES
  [7/15] Kobheerians               → Kobheerians               YES
  [8/15] Grazerites                → Grazerites                YES
  [9/15] Communications technology → Communications technology YES
  [10/15] Dabo girls                → Dabo girls                YES
  [11/15] Memory Alpha images (Nebula class starships) → Memory Beta images (Neb

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (5,299,750 pairs → 218,326 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 69
  Final class: 69, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (5,299,750 pairs → 218,326 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 77
Candidates found: 77
Limiting to top 15 candidates
Domain context: 'knowledge graph alignment'

  [1/15] Moons                     → Moons                     YES
  [2/15] Starfleet personnel (23rd century) → Starfleet personnel (23rd century) YES
  [3/15] Hand-held weapons         → Hand-held weapons         YES
  [4/15] Titles                    → Titles                    YES
  [5/15] Earth conflicts           → Earth conflicts           YES
  [6/15] Musicians                 → Musicians                 YES
  [7/15] Xindi                     → Xindi                     YES
  [8/15] Geography                 → Geography                 YES
  [9/15] Federation starships      → Federation starships      YES
  [10/15] Bajorans                  → Bajorans                  YES
  [11/15] Episodes                  → Episodes                  Y

In [84]:
# Step 2: Large pairs
# marvelcinematicuniverse(167MB) + marvel(929MB)
# starwars(1048MB) + swg(35MB)
# starwars(1048MB) + swtor(18MB)
# Time limit: 60 minutes 
# Note: Expected timeout due to large file sizes

import threading

def runWithTimeout(func, kwargs, timeout):
    """
    Run a function with a real hard timeout.
    """
    result = [None]
    exception = [None]

    def target():
        try:
            result[0] = func(**kwargs)
        except Exception as e:
            exception[0] = e

    thread = threading.Thread(target=target)
    thread.daemon = True
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        print(f" !! Hard timeout after "
              f"{timeout:.0f}s "
              f"({timeout/60:.1f} mins)!")
        return {
            "status": "timeout",
            "time":   timeout
        }

    if exception[0]:
        return {
            "status": "error",
            "error":  str(exception[0]),
            "time":   0
        }

    return result[0]

SKOS_CONCEPT = URIRef("http://www.w3.org/2004/02/skos/core#Concept")

def getClasses(g):
    owl_classes = [
        s for s in g.subjects(RDF.type, OWL.Class)
        if isinstance(s, URIRef)
    ]
    skos_classes = [
        s for s in g.subjects(RDF.type, SKOS_CONCEPT)
        if isinstance(s, URIRef)
    ]
    if len(skos_classes) > len(owl_classes):
        return skos_classes
    else:
        return owl_classes

def getIndividuals(g):
    system_namespaces = {
        "http://www.w3.org/2002/07/owl#",
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
        "http://www.w3.org/2000/01/rdf-schema#",
        "http://www.w3.org/2001/XMLSchema#",
        "http://www.w3.org/2004/02/skos/core#",
    }
    image_extensions = {
        '.jpg', '.jpeg', '.png', '.gif',
        '.svg', '.webp', '.pdf', '.mp3',
        '.mp4', '.wav', '.ogg'
    }
    individuals = set()

    for s in g.subjects(RDF.type, OWL.NamedIndividual):
        if isinstance(s, URIRef):
            individuals.add(s)

    classes = set(getClasses(g))
    for s, p, o in g.triples((None, RDF.type, None)):
        if not isinstance(s, URIRef):
            continue
        if not isinstance(o, URIRef):
            continue
        o_str = str(o)
        if any(o_str.startswith(ns)
               for ns in system_namespaces):
            continue
        s_str = str(s).lower()
        if any(ext in s_str
               for ext in image_extensions):
            continue
        if s in classes:
            continue
        individuals.add(s)

    return list(individuals)

# Large KG_pairs configuration

kg_pairs_large = [
    ("marvelcinematicuniverse", "marvel",
     "marvelcinematicuniverse-marvel"),
    ("starwars", "swg",
     "starwars-swg"),
    ("starwars", "swtor",
     "starwars-swtor"),
]

# Total time limit: 60 minutes
TOTAL_TIMEOUT = 60 * 60  # 2700 seconds
total_start   = time.time()

print(f"=== OA.4 KG Track - Large Pairs ===")
print(f"Total tasks:       {len(kg_pairs_large)}")
print(f"Total time limit:  45 minutes")
print(f"Techniques:        Lexical + Index + "f"Embedding + LLM")
print(f"                   (Structural skipped)")
print(f"                   (LLM: max 15 candidates)")
print(f"Note: Large files expected to timeout\n")

for i, (onto1, onto2, ref_name) in \
        enumerate(kg_pairs_large):

    # Compute reamining time
    elapsed_total = time.time() - total_start
    remaining     = TOTAL_TIMEOUT - elapsed_total

    print(f"\n[{i+1}/{len(kg_pairs_large)}] "
          f"{onto1} vs {onto2}")

    # No remaining time left → skip
    if remaining <= 60:
        print(f" !! Total time limit reached! "f"Skipping")
        kg_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   0,
            "Status":    "skipped (total timeout)"
        })
        continue

    print(f"  Remaining time: {remaining:.0f}s "
          f"({remaining/60:.1f} mins)")

    onto1_path = os.path.join(KG_ONTO_PATH, f"{onto1}.owl")
    onto2_path = os.path.join(KG_ONTO_PATH, f"{onto2}.owl")
    output_path = os.path.join(output_dir,f"mloa-{onto1}-{onto2}.ttl")
    reference_path = os.path.join(KG_REF_PATH, f"{ref_name}.ttl")

    size1 = os.path.getsize(onto1_path) / (1024*1024)
    size2 = os.path.getsize(onto2_path) / (1024*1024)
    print(f"  File sizes: {size1:.1f}MB "
          f"+ {size2:.1f}MB = "
          f"{size1+size2:.1f}MB total")

    task_start = time.time()

    # runWithTimeout 
    result = runWithTimeout(
        func=runMLOA,
        kwargs={
            "onto1_path": onto1_path,
            "onto2_path": onto2_path,
            "output_path": output_path,
            "context": "knowledge graph alignment",
            "use_llm": True,
            "api_key": api_key,
            "time_limit": int(remaining)
        },
        timeout=int(remaining) 
    )

    task_elapsed = round(time.time() - task_start, 1)
    print(f"  Task elapsed: {task_elapsed}s "
          f"({task_elapsed/60:.1f} mins)")

    if result["status"] == "timeout":
        print(f" !! Timeout!")
        kg_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   task_elapsed,
            "Status":    "timeout (>45min total)"
        })

    elif result["status"] == "completed" and os.path.exists(reference_path):
        precision, recall, f_score = evaluate(reference_path, output_path)
        kg_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": round(precision, 3),
            "Recall":    round(recall, 3),
            "F-score":   round(f_score, 3),
            "Time(s)":   task_elapsed,
            "Status":    "completed"
        })
        print(f"  P={precision:.3f} "
              f"R={recall:.3f} "
              f"F={f_score:.3f}")
    else:
        kg_results.append({
            "Task":      f"{onto1}-{onto2}",
            "Precision": None,
            "Recall":    None,
            "F-score":   None,
            "Time(s)":   task_elapsed,
            "Status":    result.get("status", "error")
        })
        print(f"  Status: {result.get('status')}")

# Display ALL results (Step 1 + Step 2)
df_kg_all = pd.DataFrame(kg_results)
print(f"\n{'='*75}")
print(f"=== Knowledge Graph Track - Full Results ===")
print(f"{'='*75}")
print(f"{'Task':<35} {'Precision':>10} "
      f"{'Recall':>8} {'F-score':>8} "
      f"{'Time(s)':>8} {'Status'}")
print("-" * 75)

for _, row in df_kg_all.iterrows():
    p = f"{row['Precision']:.3f}" \
        if row['Precision'] is not None else "N/A"
    r = f"{row['Recall']:.3f}" \
        if row['Recall'] is not None else "N/A"
    f = f"{row['F-score']:.3f}" \
        if row['F-score'] is not None else "N/A"
    print(f"{row['Task']:<35} {p:>10} "
          f"{r:>8} {f:>8} "
          f"{row['Time(s)']:>8.1f} {row['Status']}")

total_elapsed = round(time.time() - total_start, 1)
completed = df_kg_all[
    df_kg_all['Status'] == 'completed'
]
timeout = df_kg_all[
    df_kg_all['Status'].str.contains(
        'timeout', na=False
    )
]

print(f"\n=== Summary ===")
print(f"Completed: {len(completed)}/3")
print(f"Timeout:   {len(timeout)}/3")
if len(completed) > 0:
    print(f"Avg Precision: "
          f"{completed['Precision'].mean():.3f}")
    print(f"Avg Recall:    "
          f"{completed['Recall'].mean():.3f}")
    print(f"Avg F-score:   "
          f"{completed['F-score'].mean():.3f}")
print(f"Total time: {total_elapsed:.1f}s "
      f"({total_elapsed/60:.1f} mins)")

=== OA.4 KG Track - Large Pairs ===
Total tasks:       3
Total time limit:  45 minutes
Techniques:        Lexical + Index + Embedding + LLM
                   (Structural skipped)
                   (LLM: max 15 candidates)
Note: Large files expected to timeout


[1/3] marvelcinematicuniverse vs marvel
  Remaining time: 3600s (60.0 mins)
  File sizes: 167.0MB + 929.5MB = 1096.5MB total

MLOA: marvelcinematicuniverse vs marvel
Step 1: Loading ontologies
  Source: 1516 classes, 0 props
  Target: 274805 classes, 0 props

Step 2: Building Inverted Index
  Index built: 56209 tokens
  Brute force: 416,604,380 comparisons
  With index:  4,703,098 comparisons
  Reduction:   98.9%

Step 3: Lexical Matching (using index)
  Candidate limit reached (500,000), stopping early
  Class: 0, Obj prop: 0, Data prop: 0

Step 4: Structural Matching
!! Skipped (too large: 416,604,380 comparisons)
  Merged: 0 classes

Step 5: Embedding Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (416,604,380 pairs → 4,703,098 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 4
  Final class: 4, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (416,604,380 pairs → 4,703,098 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 9
Candidates found: 9
Domain context: 'knowledge graph alignment'

  [1/9] Stokes Crime Family       → Crime Families            NO
  [2/9] United States Department of Homeland Security Agents → United States Department of Homeland Security members NO
  [3/9] Russian Armed Forces Officers → Russian Armed Forces members NO
  [4/9] Doctor Strange (film) Items → Doctor Strange (film)/Images NO
  [5/9] Gnucci Crime Family       → Gnucci Crime Family members NO
  [6/9] Starforce Members         → Starforce members         YES
  [7/9] Guardians of the Galaxy Vol. 2 Crew Members → Guardians of the Galaxy members NO
  [8/9] San Quentin State Prison Inmates → San Quentin Prison/Appearances NO
  [9/9] Characters Killed by Vision → Killed by Vision          NO
  After LLM: 4 classes

Step 7: Instance Matching
!! Limiting source: 15,814 → 10,000
!! 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (16,701,208 pairs → 197,585 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 37
  Final class: 37, Obj prop: 0

Step 6: LLM Matching


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  Large ontology: using index candidates
  (16,701,208 pairs → 197,585 candidates)
 !!Embedding candidate limit reached (100,000)
  Class mappings: 71
Candidates found: 71
Limiting to top 15 candidates
Domain context: 'knowledge graph alignment'

  [1/15] Steels                    → Steels                    YES
  [2/15] Metals                    → Metals                    YES
  [3/15] Naboo system              → Naboo system              YES
  [4/15] Food                      → Food                      YES
  [5/15] Carbines                  → Carbines                  YES
  [6/15] Vehicles                  → Vehicles                  YES
  [7/15] Articles                  → Articles                  YES
  [8/15] Planets                   → Planets                   YES
  [9/15] Armor                     → Armor                     YES
  [10/15] Galactic Civil War        → Galactic Civil War        YES
  [11/15] Waters                    → Waters                    YES
  [12/15] Gala

### Subtask OA.4.6 (The results summary table of all the OAEI tasks)

In [85]:
# Complete Results Summary Table
# Directly using existing dataframes
df_oa4 = pd.DataFrame(oa4_results)

def getSummary(df, track_name, total_tasks):
    """Extract summary stats from a track dataframe."""
    completed = df[df['Status'] == 'completed']
    timeout = df[df['Status'].str.contains('timeout', na=False)]
    skipped = df[df['Status'].str.contains('skipped', na=False)]

    return {
        "Track":    track_name,
        "Tasks":    f"{len(completed)}/{total_tasks}",
        "Avg P":    round(completed['Precision'].mean(), 3)
                    if len(completed) > 0 else 0.0,
        "Avg R":    round(completed['Recall'].mean(), 3)
                    if len(completed) > 0 else 0.0,
        "Avg F":    round(completed['F-score'].mean(), 3)
                    if len(completed) > 0 else 0.0,
        "Time(s)":  round(df['Time(s)'].sum(), 1),
        "Timeout":  len(timeout) + len(skipped)
    }

summaries = [
    getSummary(df_oa4,     "Conference",      21),
    getSummary(df_dh,      "Digital Hum.",     8),
    getSummary(df_ce,      "Circular Econ.",   2),
    getSummary(df_anatomy, "Anatomy",          1),
    getSummary(df_bioml,   "Bio-ML",           5),
    getSummary(df_kg_all,  "Knowledge Graph",  5),
]

df_summary = pd.DataFrame(summaries)

print(f"{'='*80}")
print(f"=== OA.4 Complete Results Summary ===")
print(f"{'='*80}")
print(f"{'Track':<22} {'Tasks':>7} {'Avg P':>7} "
      f"{'Avg R':>7} {'Avg F':>7} "
      f"{'Time(s)':>9} {'Timeout':>8}")
print("-" * 80)

for _, row in df_summary.iterrows():
    print(f"{row['Track']:<22} {row['Tasks']:>7} "
          f"{row['Avg P']:>7.3f} {row['Avg R']:>7.3f} "
          f"{row['Avg F']:>7.3f} "
          f"{row['Time(s)']:>9.1f} "
          f"{row['Timeout']:>8}")

print(f"{'='*80}")

# Overall stats
total_completed = sum(int(s['Tasks'].split('/')[0])
    for s in summaries
)
total_tasks = sum(int(s['Tasks'].split('/')[1])
    for s in summaries
)
total_time = df_summary['Time(s)'].sum()

print(f"\nTotal completed: {total_completed}/{total_tasks}")
print(f"Total time:      {total_time:.1f}s "
      f"({total_time/3600:.1f} hrs)")
print(f"Overall Avg F:   "
      f"{df_summary['Avg F'].mean():.3f}")

=== OA.4 Complete Results Summary ===
Track                    Tasks   Avg P   Avg R   Avg F   Time(s)  Timeout
--------------------------------------------------------------------------------
Conference               21/21   0.370   0.566   0.433     326.1        0
Digital Hum.               8/8   0.693   0.225   0.281     202.0        0
Circular Econ.             2/2   0.307   0.918   0.450     143.3        0
Anatomy                    1/1   0.787   0.741   0.763      60.1        0
Bio-ML                     5/5   0.351   0.043   0.076    3416.0        0
Knowledge Graph            4/5   0.186   0.052   0.074    4999.9        1

Total completed: 41/42
Total time:      9147.4s (2.5 hrs)
Overall Avg F:   0.346
